# Kaggriculture V22 Safe Meta Edge

The notebook keeps the complete public v22 Price Impact route unchanged as the anchor.
Two narrow extensions are evaluated rather than forced into the submission:

1. **Terminal liquidation** on the last actionable turn.
2. **Public pressure ordering** that only permutes already-existing SELL slots using
   recent shared-market supply flow and the opponent's visible production topology.

No ordinary-turn BUY, HIRE, movement, quantity, or SELL-slot count is changed.
If the extension does not beat the exact anchor and survive regression tests, the
notebook packages the exact v22 artifact unchanged.

## Why this direction

The attached v22 notebook reports that replacing the stale complete route produced the
largest gain (`1/46 → 36/46`), while the official price-impact SELL ordering added a
smaller but still material gain (`36/46 → 44/46`). Therefore the production trajectory
is treated as frozen. The experiment targets only two remaining low-dimensional edges:
terminal cash conversion and collision pressure among existing SELL blocks.

## 1. Exact v22 anchor

In [1]:
%%writefile /kaggle/working/v22_parent.py
"""v22 price-impact route agent for Kaggriculture.

The production route is a real fit-only trajectory.  Runtime adaptation is
limited to identity-free weed recovery and in-place ranking of existing SELL
slots by the official 1.32.4 price-impact curve.  Ordinary turns never create,
delete, or resize a market order.
"""
import base64
import copy
import json
import math
import zlib


_ACTIONS = json.loads(zlib.decompress(base64.b85decode(
    (
    'c-rk<U2hyoa{MoR<^$)06zMmvG-nCN6$MJV!FfR}7VsGcjPt|VZ^r$1Yei0XPiJIgWLEVowYs+fIn!O0Rb8DK85#N0|DFBEFTei%'
    'Z@-@X%TH$?Za;oJdptk;&tLxg-~Z>!AHID2`!B!#*Wdp4%jciY-oAU-efh8U;fK$E{rUF&yB}}w&d$$1zTNFUoSm=DKVIMOCx8C9'
    '+r9bn$Nk&g?WeQzSF=C=xVyW5e|ElFKR*8B{AkqgUjO;?hso83@&9zT-+lb_bv*CyA3nYP`ssO+liy8;_w<9~iT^f;4-'
    'fZmKEM7n4$lnZhtKcs-'
    'u(RK>YqM;wZUW*<IUMHh6|71n~q~X>$}_6yXQ&kH#7evcX+nl<kIsg!dtjsBDW&88&>ekgx`<$KQ`gBEuI##(SC>fJngryd*b%f?'
    '&0{3fBJ1tPDl0p?UXsk>yDE=+~D=?v+<t4)X8|`q=q{U-'
    '?3XioPk{vU@N;JW<TSrbaVsJdUiv^W;|RkX})16G?<TWwP7ddTWx+_(Q4yP=wdAVpw1^8S#ADK60J7*lWsP1t4`JebMP%<{xx~H3'
    'dRBk@ogkLkYp<6Lnjm25025gjeBORZsR`waQe$WpCyii2mPFl>uwESNL|nPP0t5t(>3Nt>+dy=f_=?34ldOnVluni_J!#&j@S2hc'
    'e~f0e*V+$;nTajcmH<$@~T|%<NfFMW$HhyHxKt8mVKH&?(TjI-'
    '6lgGBe+F2M0f(N8n5?ao;YUs^3KWZ+g>*TF>P`;sTf0HbvY^!M~?HAo?d2j*7fVn&$pxNp%pM57BuPja5$D)J%$0wI1u3fTA!}r-'
    'qxt26K0Ltb=pn#kByKx9CHvsY=q3MNkCUx`(D!qVas<qZ*Y<<G;uc~>U8h969A_>e0ciya<~2tX3byZN-VsX4alwkm?kKM_Am9$e'
    'Xsv5U2W#yZZrPvR`qYWqr1h~G{v)0l48#nQ&30dK!ID#Z!bhjDOWXl%QkZzWsz#${v0LkZ7UQ2F}HH|zsfDG5$%jXP7)rpRVSX@u'
    '`tEVn~c5I>o+7e4I%g*u$y?lmZ)&k@ui)3ktGIX(39U`3@^?oAhG$~+XNiC|1?Uktk#Ph!Z)25T<WrNuF&(7owt7%5Bkz0p7rsdr'
    'vX}TADZ&G7ILBorc09(O{h){QvqIe3?VLOJv+&fCZsq>32>ZI#~M&_!5x(17Gb)Z#Q^f*e{XLe{-Qd<8v!YOcK-'
    'R)byCeRJa`Wj&(1e@m3L@|KaNGAv{}<ke~$xZ%orqhLB5o^%wR4l$`inFW~BN0XY#G%pQfL{rK3R;C>zlTaVAS(W)TYJ;eOLaw=;'
    'oHUuFdeqtHvx$<Lm(;?Wae4LPn^Y`f3{HxmOp;+fJh{4cj9h6#Wx&*v1L(lA?}oHO11&T_^xPb0F2SIng|uZ>^MfN@rXoSFxv;4l'
    '?{#MG<c=_09tp(RQUGmIMC*f}-MG#OI(i3j?n24>S*;lPqMuNcs+hX{x3Vqy&kV{*bC0@ZRfS3~CS<tHtn{bhS-'
    '4}X2tH+0|hkJwKZr*}(n*;mA(L3tkou?bb|!^Vva)RBxvkdw}gpj@EXm7S5JVc8Wt-L&J?-'
    'Xt<7cG?U?($A1hE(xqWj2To!A<iX27e5Ls4p8A5HL<s{{)-'
    ')JGF;1`4)%gdgs#VR&E~MSCM&mZ9XPqRVi!Buh%{U^%iKM=3}f8;`@63d^Ekx4n*6=ue%;-'
    '^f0Hlvj{~=PJwNt$kh>tdi8r&v&&N*>xBDM<4-bDmJHL+K$g~XJ&;Bfzwwbe##`D*gI7o^CcwsEQ-gq=-'
    '?x9$EyfXOTVIV^WWcHn<t?gBv`@mNC+F^DdE<Vr9p)AB?kGHM?D0e~c>b|sD;J^S&fTYP_iG_LO$k0avVVikAgHEFqBN<OV<M?C?'
    'Hc|^W+ibdF9fuF|1Zt3lX>iKrr3DpA=m-?R^(|rfCYID%4IRYUg-'
    'i^NVIrfOg{=w3EUfD#VRsahG43j(Povi>1mlUp0tcSFRHx<}M-'
    '(3=9j%9UerVBo%+P79gpg>mcSXKe&wXdd)<i|CnWIToLY*P0wbJj#sFiZ)bs`SiRf(y_h0vza$~F;BO^<W<f)|Cz8u0nmMf}DXrL'
    '}CCKOT7UlMn@5KZ==C=T5^MfIr$cvYSUeKtOjIhE2a!;m2i`*2t&L-'
    'w?aGBLl|PWsqcHL2zT&)s7}`U&L`ykpUJ<BE>Mzx_~^l(1LUCGo&X)hL$r&7ulCHk8m0C;SRXdK856~iF>o-'
    '?lxvQb+IWxv&{pK7*7uI%8@|}tB$mj9da+hs!u#6pN*z5D}`YiAC9o_oJR+pn69V*&XkstEMN<=Y%FCF7s#y|Ay7C)!J^v#wWXoB'
    'CLdztRz@HvDC~0ud7YBzXJuG0g!Jz!)mwJW;4q506jC+k!IC#-V+o<>xs({cuDdhg&7ZCf!>1qL-TfIbdEhzSZ$Kohrfk=ok)rUF'
    'q$0@x9gF4Dh0ibaO(|&QqTbV>^_7#4T;+q=wHnN>nu6J<4>baq6;x=6Vds}iy&Gp*#W=pog{;No;jBg}0Q&olgk&0QKID+#(_&^P'
    'XEI)?wp75K(qkn$-'
    '~x5oE*n`$KT+dHB?Z#94aH;ofTA%uRIcEY^wvCzB#T<{vA~Y7OBkjy#XNJ~<Ixh9nFZF95iAEA?m|@6;^S$m_uA|)qGxcIdl(zxb'
    'AnH543b5JuZ7(dZnDr>ON|ivw}_+es-'
    'Z8jdJ#M(EmP;UY|>!qH3;JOFE@r6v2VMa;72brwMfI9Y7c*(R%w>;&&G&KG|jReN|d)jf4tNKJ`8ZLGm)R=UgGp6NR2GS4rPw!Fb'
    'bN^Y#5HB1j4eDeW|mOh=q-k?O!+zxx7-'
    '=*&i%oS4!qwR=I+l&uiDpN=Nuvk0lDEAW#nWx|l)iNmWG#1fC>XTVIT$f#V2#C^|5DTBQhkn|}IbeBo_a3%@Vz&jbaM#b6X<V~9^'
    '*3aMNWgH9N503c3XSMH(sK(WjtJfyk~QWX+GpDeEgWW^U7CqKZ)kpLfoVNvrjUg>4qnvMw?Vvy9Gec;1QJ~jal10h7?VTH%d(quz'
    'B)bl7T6eWlv{FH06#CtY=sKLIH7j&+YBUx`+0r!+eQ{I&J=23z#_vc|MEawL%aQIWAawe>Z?FP|ExLK0E+p|{k*?OA`Csw^FB}KI'
    'o$*99I0Gkl0m)9P`|Dt2CN>`qyc-LXmu5Cd2o!SUN>OzDV@*3+rW<hAgD?;=UTjm9@iL;1$p|E*c%;VQQvPe~$5So8wMuO;r<t77'
    'b(uHBcTY4h;_KMiPAfF!BvjzMoFfYFtu8;UUr8Cm{J0zsZS_lW3V8=F+o8BZ#GAtykeAa2yG$0`v`_X2mq7SrK?pnM$-'
    'RiSbCt9_TfNotPdP3FeoipHvk8_bHdfr%D&Wevel&c4bQfUJ?0lZZJea6gnmC8$PxKJf3Ae^ck41GgWvUhmE`|wJq5ML9PJTI2zv'
    'S<Ux={f<ZuN`z*9Y!6J==9)O*NHS4t=L!Y^j|$z3H;H6;q`>i5F{w6TC0y}FvWZZj=QZs3=;i9Cvp+{##mZrL)dzWaF+^NS(G_5&'
    'MAUy6$1No&H$~^%)k>g(OT6xLX0rx0XLmi9#>)&UkW)E(rDTdd&?g)nlWZ@qii0%$X<Fb;<l%9VOwtElmJ31UZ`$(expkjBMXp>0'
    'QjHVtDOg)>d2b<kkFP`M2F!}7IV%(^P~Eci!N#n=^~cPB6w>VMkO`8)wWW(FCkjJsZ1918i#>u&E+DXDk5QDk|15`lPCIsv21ZHk'
    '*GjjT31#kf+Jl;ht4Y#dXG%!p@nplSvk9p0!~W{)^u4GLCASyvE>_})GHlQ()81{@+s0-'
    'MhCr8^4CoHq^6m+)=dX}a;Ab}!w1k1s*}~2CW22S&8bZACoHjyxkrb|WMrCuS?_Mlv@+iw;eW+a`r&=xGYkNpr9Wn?rZ`d=RKngZ'
    'BH4Va#yP?UA!<ncQXYwrsE{XnsF)=2`8R#H1Qk}`FRP?>0I}AL6vPy1NXgK3On9oa=1E6{Yz?W+(!3rU;LVPYZ7%3bM|YcibhnC|'
    '(e#ij$Od&;dP$Si8YM>eB2lpL&1M~2@2{)?2urA#?_PgDNcJ~x0c!ost5LB&EcUj0!-J-'
    '4w%1DXiT_n?VoKB5<WDX`p;?BC4kXEa$n<w1x23w@J6Q=q($QzkqQ>d70(7*s089)Fz_gxAa$9~WN2qdGH~=UegFkS@W`&-'
    'a=ki!61F$_lU|R0|<w0m+fYwyO?X%FMGmoSxgBwm4Mx#>^NJD5{jBx=o0g8eq^UEQ*9yXJe&x%zj3E@E)43Z@Gbc5xCpT-_fTP@3'
    'V<E6B0c~DJ~T*w-'
    'ARBKzub^L8&Y|}t%?(#3zfzmCz?^9S$YjkJPM4!Y{oH@xDbzQ^ly$bn0X(Uz+KyBKwz@64n@ZDRcAET06Km9OIdM;N>O`?%hE0#_'
    'jcbF-<(xPcW1an>EUD4E71pp`3bB#Aef-uWuGlC$!A*p!vm{Go_<+bsOB!(fCr@Fc#%W*bQ{Hkh$Xg8J0TJ1giwIbCy^rc8FEVl-'
    '4B*sep>in(iZJl!dIR70Tp(qZF0JX?ufbdxWPC~f>vEZU-'
    '7+^}3tx^AoIgaI$9b(!e&%uM^ZO)A#pjuTaPUH|;Qz$1IW^1NJ>SP;`*%nX8sJRISVfR^adROL!WRS%rlRK&hNuje-'
    '<ba4{LziVIgP`qsq7JpjCrBr;SpgqQitoDToU*2*<x3s2Q=Xnu@RU<6q|ls!LNd+P@Ekfzr}&AuzG<FxE{wJ`9oRyWK$k9k#EvBJ'
    'h_GT8TBejpVqrL+3@=emNUe<<m(3>Hg)2xiznaQCu(4&7g*|3A2|x&JV@NZUnu8@3qO#jXPxQ&kDAH_}P-'
    'K@jK`xjF4{wjDaK*%ErGelZDTrI3F|f?HZ4f+bAX_%Kwkj6ft=2RbdHV9YwV=i={bt)LB))y@lC@lig>OqMiPKp4sA=F~=+ljG=}'
    'd3>oy~GdQHx4gZz(XKtuSrQ?Gce1h_*>2KsDP(mKs1ecBl#=v%?xXry&{7@XdJ?IQ)6}tSwbcra(np%GFWzIrM=L-'
    'XxHemrp(7s29W@$Gc(rGjT&?#!G6!Y-'
    'J7b>xs&j>AcFvoC}!vC#tl!ZxH(5B&z!`<&QBYk5?Z4y|vPh^r@~PXjSx!fQ7jh2nhzU13ZB)7Ty!6rwKD<e?~Y77F1?SLj|I-'
    '3;<E^DQ1(}*t#5WK~N6zjHYX7ewAyQr7TM<TL(rNb{O6y&qX>g)@(SL`n5)4T#C>(zLHtSHgd~|2p>TrRuYN2r`GpT=^uzxmo5q1'
    'c1oA8Ut+qPe}XXdGAVH<u8i}{y|bg{?VD{VJVCh7oK_2@!kJq|6R3UC-Y#v<?KefKu_s7#2i-'
    'rk_^6WC*4|6Br%x0YcS>WE9z2fRaw3HZJaVO84UZt~+_P(F!AvVEcQk22%ZO60AY-'
    '~~y~+yKO1{?#rouhcDf=1XKeW2CvZfW?bu?a_A*<{=sqZov3wq6~)((B{tJv`fB1wK|h2zh)DAF^uT8%zlXCkzk4`Os8F=w`s<|6'
    'FHn0><Y$xOXRlrr*`>QMPDX2xOj4M~~tX{#M8>2lQx`h^ojpb|V|!IRq3AvwZCns^=y(+li+3l`V2_O;yaQM;yC7ATz!o?hQmTzv'
    ')h?g^Oc<r35d`RQ_d>s;bD&SjR9*KIYgaSC`XrHNa6M3-'
    'tnlaV|{zoM!4F4f(+{c+<q*J52n=M*&SM@*q0%Bftay}3j<2vjM_E0l_jEL3%Db__~VDpDQ3b9|y4H?QjeX>B<&^U@=I8watPNAO'
    '^>!~yeI-F#)IX@DSwoUyzl!;AhH19<U80sDu^n4xWHvmy?SL7-'
    'VAN_lDD5xFa`Smq(nJp<WKZBljZD|9#<n;+DAdeW}#nl$N><5)7uZ9dT#7m=ELg5NQ#78tsE>uqY&kXuw3EX(^+w4c=|p*DA!AaT'
    ';A=D47EEoIN)MoTmu6aA#rl-zj6E<hA-Q<+Fv(JH@lTC?@wp(>T}2en<fJ(m!+U=5sza`Nz@wsEF_#&$8)<oLjwerP^*(%gw-'
    'D`W}%O5MRf`4e%8{9+Ie%x8<r1KHxV2DE@=x9n3+;&1xOY#DK;`3u~uSxyFdXD-'
    'w+ov}8Kxz(UB1k)u>2h(JeSiw5b3jIuPmL|wXY6OMz_7VQ%B2k^1eX}5%>ffbUNoC%8lW3ikRmD+pep&h!jR5UCO^JZc8GNe*IW_'
    '6$NoY3=nc<>nyoD1q5LXL1bZU8A$FNvWa9%OaBH!Sbr4_m|aqw!J==5zNc0Ce~j|aMPei!kuF8JE^OnFsrVoqO3k+uT<D<jA$&ca'
    'r}Ktqa~TY<p&Cr!<cybL>{xA$%K1K_!6?XU%jaz=pSe0}`3ebUt&?*%JL`vFwiWV@vZtM)at>=ReaD8c$GsfUDf(M25#3NN}mq|j'
    '@+0Up;<Z@XA?Bg+h8IMCBKDP2^C0LhTOZKFePS6!J&t1H}-y;-'
    'Kht1LQ&w&wc8V3*Hf;l(&tD#M48qs$d0>~B>tNBK=d#2rKuS5N}9X{Lcy$x%e9QfPqPr>?9r6T6EoxyWI*0w5@^G_SYOt0Q!bCB@'
    '6BP%r`N#zHZ#K42Xy3$vRnmObYJtNSk9inoak=Wz=nAQF1d$CuFA(2`ncLvSB?MV#-'
    'mQv@d|4wcnz?b@P%?O=+iv62P0bQb<tPU|0bod6^_E*$FpBn~1)Srjd^q{>N(qN~xe+OZQIkz9!%5m2EOU@R)piemYLNV*jZ8ajI'
    'YSF}RMzl-17wYI0|tsK<L*d5?bvD~jyFC=6_MigI!NWw07<x#=qM2RK<5@Ibdx41moJbSGS9blPN!?QDmnmcCBLhz%YffK-'
    '}8fQw7j1C7sH&KtuUeDo|I9?$TmHaw|BW(?-UZEb)WQpDZXH7!KwXB#^STu`Ac@H!J-'
    'DF@aqme{bgYe&MX=zbeR!05w)l%~n+s8h=FsGOHiX0Cy^hFN_+vQ0ZZ0p~h!B(!)TZb8jWEEV;t~#`wQe%Y)f9lp!$HE<4N`R}wV'
    '>O`#TI>J~R8ba&Zkx_McEAn@c^}$9TY7~cN5*ag)tpK%_1dxrf{GBe;wFs{NhJzq#4>1Exq){ifP?>^*c0ipC7HEEn%9SrOllL0('
    'I}ftOdJTlLHPsv_C}AjBOxqV-UA#o4RWcgMXc2uv9{$WFe*(h%hb@a)+nBg9zh}&l#)~?T|yhgmN*$j9D7*gq#wtW*3XDASu>kv0'
    'e~aTpJVtugjIV|HGnqJ1i)sO#71+Vvy-A~Bq1xLd$|p$e%#&Nzeh~SbVU?A;8``ht?&r-'
    'oT4~a5LXKQNn2h9U%&Qi^R~VCPwxN1v2Rv(-PY-45^A&~r~tq&YWm;xu2U{QDha}w{QxwP*nXk}a<epbj^&o{aEF;Fjtjw};-'
    'zY$Zf!Ah%Sq8L$LZRfgwwTonefjl!E*7LORPInFPhe6Cv-^Am-hejVrg8o9*&qL&*MsLqdvSYsVYwKRnsi4R2jl!7vu`e(Q$TJ{S'
    '#a_KW*Q2+xb*2k0GLU&h=}}YDHO_$EGgS$s$t9UZP=cjlP!QG|@iQOv2ZLKvdb}242(UV_V~EDDwtmNviimIzjc>rLTkv3#}Qotn'
    '!`aK*j5Lo!F<+I;tQ`yt+PI!WR(E=K3+w=!tV~qQX@A$z{}q%))BV?uKr`@hK8Ncr_ZGcHT*p5GgMsfUu=T2|*Eh;tXe{O#$g6ju'
    'ufra;XhD+)1hg9D5e*L)Jw#&D+~yy%$q=qz;FlywD#Mg}K5{U_YG_&6nXZG`%@PF|X6IW-{Uke}oel-'
    'm0^?3B?MlUaD1klP{eaMc$}TMK1N|NfbBjeA1dpy5N}|e<eYXTbFP-xHqIXmP<~PB^5*!`cSUy`O=#UtTaD+_#NWdMF;fo@+2r>X'
    'PGk*@NAbq)5u|Ebg&8e1Da<P5b^=8Y~^1QQ_L3s1)Zeo6^c#Ufrz4qJcPD!<~j_9l4cAg$SS#)*kOSKV(}VmUs4*GVH#*mF~ksz{'
    'ind8XC+i}=*oy=bPKYFQs(4&Qz&=$mD^DTZV(W4)x?G3D&KhB!DIC1-'
    'M@OCBo~(fbJJrTQp~ICN!v9}C?rPd<_xJ*?F+57QLNj)@(MhxKYRWXd`JyYsPJT2t7~su0W~JrB4Y5>rHwf=|6z&3nc)M;=UElyq'
    'a9#~&4rf*=E5;f(8z+R_>2$c8n1*<b5u~YXZVz;Ma76=r`1je%&ZDC)sp?XDD(@eGyw_)76b^O;nS$SCEe<#g{j7r+9D!+-'
    '+p_sS9v%d!x#BTfHXc2-(oETp1;UqT9$3<cf-'
    'V!+M$)zT2<R(e6!`0x4Lx=B;Cd!=&*JFbJ<GUTJuhYCso(7#ZYP5*7`+gQjsv9b(HEt%W9j-'
    'rlPmJ&y^a`?m)_3?!&@@I1p9I(mE4Tku<K_+pMcadi^n$53NiT6u_5P)=ITflvz7u2M1MWX{B#jHB0N2zR$H*r_G!hA5fj|s*PsG'
    'fYvCsleGASjk<2)CMB9wBbuY6z>!o)s}bt7ng#Y?rn&**)n%x-'
    '((_#<(dDGIoMICc0zNzTVyg#fzs7rzg%gp*bL^N3{HbP)ov_zpn{3Dqn~CD!Kr4pMj9DOYsID$DWI<<Q43y+&ke0L{zpS$cJ;nvE'
    'u=NU<0wl6lW~-'
    'w}(T*!HXy~8XG`)DHAS$aRWtutFQm*n<VbmPXKtT;Id^bsg`^qAfO5l`diO|lI(qsDwQck3_=6~YhK@2vBbq3KkJN}(rR+4*^gxC'
    'xtslCr`F6RL1%LACy8bR!;5bou5UtRw-'
    'CwvX>P}9+;10Z?z`&EQ)7`rt;Hw9ExIvW})RC6(DLvzeBIZVPA)5U_zvBCf_XmOo9D@-ZbuMvY-JljAI6S9XjnwYd9g25#kBV~N('
    'RxFv5CrqHq+{!+CgXV_=7B`8^<T|N@Y_%l+PKrH)q$LFfT!bYxU<P{f9Zp?cyiCJp(sg>NGB_Bs-'
    'QOW6h_z%QTU_5Js1kE0lE`9#!h=zn5S3UONIZ%wp}aIW$`aKHI$_S%AGpw9ZkOf!YY(50o2;1t11)i4MyP~tC-Tb7j@qk~(Op#S('
    '?dm^9z1ars?I?lxhzY)*JEYoAR(F^Itm%7g}0{GtE$3Eq`U?mRY=_L;+~7mZ4=S>C<%;l)nax|*d~0&v8h3Il+Pxso`T;NSZr72)'
    '|TOZlVgRXf?Oxop>4m|PoG`nLtA2Yfh}g|Aq&p$eWg7160Z##V@6wp25q{6Qrk6#n;dq~g3P$fmxEB9UbCz>`R$t&DqDqot8Yq&E'
    'Ikt+L|967OrpEh=dldija5W4sv8&lNzieY+Exq_mEN6HnoDoJw<RxtlE4^meMlF`(obT@iOp@J^W0dqWgI_-CE0R8Xei#NHig^@<'
    'S)K1QomJ^<dsHu()w_b(vzU?N7jV#xtu-'
    '>T}8zRkqyZZhEn!r^{v$RbI?}Ujy9B(q&3DEY8ipLR96p1bxJ^sa=wKSrX&=srZ2_{5!vg9@vyYb#lBVQT!<S=3se)E97uN%VXIu'
    'kO6__`>I0fc0Gf=)*l7W{jAdg<x+pE_+sA#VQcpR{ywE<I{28W2CbaIPhwxk2d55ybG;LW%Q7WuMpshy-JRJGq+l9_Hq*HBTidb0'
    'cDbNAo+e}ww0Z?+LU8LIR#5TOFwE>VjBmh&$jUo@>>9vDit-'
    '7LeHwKh@g*yv?x{pSD?)v4l4(Lx5Tc1QGtSl$rkrxKJR2du&Dx7u3<O&$cP?_GL(p0J0Y}6DZj#paVU&}W3))-'
    '^YgQ%X>HeALU3zFAtlTc(W6=XGasDz;!hslCEvxK;INh%wjES89o3#ou`O32wuNv9<=RGz0|2QVxela%HH+->VYFdr5j2g-'
    'p`Wpcz|sfm2kU^x;+3CaRceejTd%}dz<G&t%mTLXBQ#rFiGYFq(IrRCJfu#M|3r4gNDTltSGIkKs3HN&@w*=ewOfIrc_&c?2X*<5'
    'QP>R`H0ky@x|GwWPYCAr01JiP%dVV@`C)3E*M`XY`40=FU|y1lRR_F#k`vUT=JWqaF!(^7OY{UX~{!nQUp-'
    '_}(ZvNep50cTGDBJ7qYV;UX`$k-'
    'JsLRq8~6%X3^rKI5MLU(q`u60y#fMG82(F@*A<B}g<l#5MX6v$5@VVjjwH0h#pTdb(xQSR&y!AZ;GI?c6?oe5>hF)oTxDJ4EMy>k'
    'Mwd!<QOeT;=j*4k>+KbQ}mm9%H%vQC9MvBEm4dX*96m+BO)G_^xjq1t3(ooE9blOtLYDgq__0p~ckXL)?)Z(yyhrlJN{<kMKw@yv'
    'l<phq>%vRI>$wXpj)4AEp@LlXTcwXWnWQ;{7R)xwoF1TqdP+6vI+=n$cdi0r1=OS;ezqDZ#_Z|qS+dXQNo3{7nbrWNUSFf^J(0E4'
    'Q9rJ0=!FhRLENyPrR&OV!SV?RO}R1-C|0GopmQJ<sIb4Z+4h!1cpDMCe5B|}r^*ZUag;6-zVDS<pyO?Zi6oKxnLJNU7-'
    '072m*m>$QcXh%MSHylefSvJg7)0T|@F@vW>XsAxR0)!|2o3f`}u=EySSlG@t;BmTaf?w>WO|;-'
    '*TC#@zB7zhaaFjrYwze64TcpNWkV<Go;P{56SvM>C?kllLrp-sVA#;{RMM%X?t5Tc$@NK8cB_{&ajCt|Rh=bZAQOGrTGTd05h7t('
    'e>aBX2rccNorthpCSP92%%Wpl_QQoEuiQCL8?IgZbkg5_{$(&x!U&M-'
    'E3qmT8M53(<quegI1<6C&WeSr~H?Fh*pi0Mu9e!HO8+I;Bcp1Al%O&efQ=Y&+L`I`Fwr~CVHV66mkH`N9*sTax'
    )
)).decode("utf-8"))
_PRICE_FLOOR = 1
_MARKET_PARAMS = {
    "WHEAT": (25, 10000, 400, "sqrt", 0.8, "log", 0.2),
    "CARROT": (35, 10000, 450, "log", 0.2, "sqrt", 0.7),
    "TOMATO": (60, 10000, 200, "linear", 0.4, "sqrt", 0.6),
    "STRAWBERRY": (120, 10000, 100, "sqrt", 0.7, "linear", 1.6),
    "MELON": (250, 10000, 300, "log", 0.2, "sq", 3.6),
    "EGG": (50, 10000, 332, "linear", 0.4, "log", 0.2),
    "MILK": (160, 10000, 122, "sqrt", 0.6, "linear", 1.6),
    "WOOL": (200, 10000, 105, "log", 0.2, "sq", 3.2),
    "FERTILIZER": (100, 10000, 200, "linear", 0.4, "linear", 0.4),
}
_WEED_STATE = {0: {}, 1: {}}
_WEED_REPLAY_STEPS = 8


def _get(value, key, default=None):
    if isinstance(value, dict):
        return value.get(key, default)
    getter = getattr(value, "get", None)
    if callable(getter):
        return getter(key, default)
    return getattr(value, key, default)


def _copy_action(action):
    action = copy.deepcopy(action or {})
    return {
        "farmer": list(action.get("farmer") or ["PASS"]),
        "hands": [list(order or ["PASS"]) for order in (action.get("hands") or [])],
        "market": [list(order) for order in (action.get("market") or [])],
    }


def _seat(obs):
    return 1 if int(_get(obs, "player", 0) or 0) == 1 else 0


def _farm(obs, seat):
    farms = list(_get(obs, "farms", []) or [])
    return farms[seat] if seat < len(farms) else {}


def _align_hands(action, obs):
    action = _copy_action(action)
    expected = len(_get(_farm(obs, _seat(obs)), "hands", []) or [])
    hands = list(action.get("hands") or [])
    if len(hands) < expected:
        hands.extend([["PASS"] for _ in range(expected - len(hands))])
    action["hands"] = [list(order or ["PASS"]) for order in hands[:expected]]
    return action


def _tile_at(farm, position):
    try:
        x, y = int(position[0]), int(position[1])
        return (_get(farm, "tiles", []) or [])[y][x]
    except (IndexError, TypeError, ValueError):
        return "LOCKED"


def _trace_actor_action(actions, step, actor):
    trace = actions[min(max(int(step), 0), len(actions) - 1)] or {}
    if actor == "farmer":
        return list(trace.get("farmer") or ["PASS"])
    hands = trace.get("hands", []) or []
    return list(hands[actor] if actor < len(hands) else ["PASS"])


def _weed_repair_action(obs, action, actions, step):
    action = _align_hands(action, obs)
    seat = _seat(obs)
    game = _WEED_STATE[seat]
    if step == 0 or step < game.get("last_step", -1):
        game = {"last_step": step, "active": {}}
        _WEED_STATE[seat] = game
    game["last_step"] = step
    farm = _farm(obs, seat)
    positions = [_get(farm, "farmer"), *list(_get(farm, "hands", []) or [])]
    unit_actions = [action.get("farmer", ["PASS"]), *list(action.get("hands") or [])]
    active = game["active"]

    for actor, transaction in list(active.items()):
        index = 0 if actor == "farmer" else int(actor) + 1
        if index >= len(unit_actions):
            active.pop(actor, None)
            continue
        age = step - transaction["start"]
        if age == 1:
            unit_actions[index] = list(transaction["intended"])
        elif 2 <= age <= 1 + _WEED_REPLAY_STEPS:
            unit_actions[index] = _trace_actor_action(actions, step - 1, actor)
        else:
            active.pop(actor, None)

    for index, (position, intended) in enumerate(zip(positions, unit_actions)):
        actor = "farmer" if index == 0 else index - 1
        if actor in active or not isinstance(intended, list) or not intended:
            continue
        if intended[0] not in ("BUILD_PASTURE", "PLANT"):
            continue
        tile = _tile_at(farm, position)
        if not isinstance(tile, dict) or tile.get("kind") != "WEED":
            continue
        active[actor] = {"start": step, "intended": list(intended)}
        unit_actions[index] = ["DIG"]

    action["farmer"] = unit_actions[0] if unit_actions else ["PASS"]
    action["hands"] = unit_actions[1:]
    return _align_hands(action, obs)


def _shape(name, value):
    value = max(0.0, float(value))
    if name == "linear":
        return value
    if name == "sq":
        return value * value
    if name == "sqrt":
        return math.sqrt(value)
    if name == "log":
        return math.log1p(value)
    if name == "log10":
        return math.log10(1.0 + value)
    raise ValueError(name)


def _market_price(item, inventory):
    base, equilibrium, scale, below_func, below_target, above_func, above_target = (
        _MARKET_PARAMS[item]
    )
    if inventory < equilibrium:
        amplitude = below_target * base / _shape(below_func, scale)
        price = base + amplitude * _shape(below_func, equilibrium - inventory)
    else:
        amplitude = above_target * base / _shape(above_func, scale)
        price = base - amplitude * _shape(above_func, inventory - equilibrium)
    return max(_PRICE_FLOOR, int(round(price)))


def _is_sell(order):
    return (
        isinstance(order, (list, tuple))
        and len(order) >= 3
        and order[0] == "SELL"
        and order[1] in _MARKET_PARAMS
    )


def _impact_score(obs, order):
    if not _is_sell(order):
        return float("-inf")
    item = str(order[1])
    try:
        quantity = max(0, int(order[2]))
    except (TypeError, ValueError):
        return 0.0
    market = _get(obs, "market", {}) or {}
    inventory = _get(market, "inventory", {}) or {}
    prices = _get(market, "prices", {}) or {}
    current_inventory = int(_get(inventory, item, 10000) or 0)
    current_quote = float(
        _get(prices, item, _market_price(item, current_inventory)) or 0
    )
    later_quote = float(_market_price(item, current_inventory + quantity))
    return float(quantity) * max(0.0, current_quote - later_quote)


def _impact_slots(obs, action):
    action = _copy_action(action)
    market = list(action.get("market") or [])
    rows = [
        (_impact_score(obs, order), -index, list(order))
        for index, order in enumerate(market)
        if _is_sell(order)
    ]
    if len(rows) < 2:
        return action
    rows.sort(reverse=True)
    ranked = iter(row[2] for row in rows)
    action["market"] = [next(ranked) if _is_sell(order) else order for order in market]
    return action


def agent(obs):
    try:
        step = min(max(0, int(_get(obs, "step", 0) or 0)), len(_ACTIONS) - 1)
        action = _weed_repair_action(obs, _copy_action(_ACTIONS[step]), _ACTIONS, step)
        return _align_hands(_impact_slots(obs, action), obs)
    except Exception:
        farm = _farm(obs, _seat(obs))
        return {
            "farmer": ["PASS"],
            "hands": [["PASS"] for _ in (_get(farm, "hands", []) or [])],
            "market": [],
        }


def _kaggle_submission_entrypoint(obs):
    return agent(obs)



Writing /kaggle/working/v22_parent.py


## 2. Candidate builder

In [2]:
from __future__ import annotations

import ast
import base64
import gzip
import hashlib
import importlib
import importlib.metadata
import io
import json
import statistics
import subprocess
import sys
import tarfile
from pathlib import Path

WORK = Path('/kaggle/working')
PARENT_PATH = WORK / 'v22_parent.py'
PARENT_SOURCE = PARENT_PATH.read_text(encoding='utf-8')
EXPECTED_PARENT_SHA256 = '62fb5a5f66f0011092a2b51e3192879ba583d5d815d761f176091c615657147a'

assert hashlib.sha256(PARENT_SOURCE.encode()).hexdigest() == EXPECTED_PARENT_SHA256

OVERLAY = r"""
# === SAFE META EDGE ===
import copy as _edge_copy

_EDGE_BASE_AGENT = _kaggle_submission_entrypoint
_EDGE_LAST_MARKET = {0: None, 1: None}
_EDGE_FLOW_EMA = {0: {}, 1: {}}
_EDGE_ALPHA = __ALPHA__
_EDGE_BETA = __BETA__
_EDGE_TERMINAL = __TERMINAL__
_EDGE_EMA = 0.72
_EDGE_PRODUCTS = tuple(_MARKET_PARAMS)
_EDGE_ANIMAL_PRODUCT = {
    'GOOSE': 'EGG',
    'COW': 'MILK',
    'SHEEP': 'WOOL',
}


def _edge_market_inventory(obs):
    market = _get(obs, 'market', {}) or {}
    inventory = _get(market, 'inventory', {}) or {}
    return {
        item: int(_get(inventory, item, 10000) or 0)
        for item in _EDGE_PRODUCTS
    }


def _edge_update_flow(obs):
    seat = _seat(obs)
    current = _edge_market_inventory(obs)
    previous = _EDGE_LAST_MARKET.get(seat)
    flow = _EDGE_FLOW_EMA.setdefault(seat, {})

    if previous is None:
        for item in _EDGE_PRODUCTS:
            flow[item] = 0.0
    else:
        for item in _EDGE_PRODUCTS:
            positive_supply = max(0, current[item] - previous.get(item, current[item]))
            flow[item] = (
                _EDGE_EMA * float(flow.get(item, 0.0))
                + (1.0 - _EDGE_EMA) * float(positive_supply)
            )

    _EDGE_LAST_MARKET[seat] = current
    return flow


def _edge_public_supply(obs):
    seat = _seat(obs)
    farms = list(_get(obs, 'farms', []) or [])
    opponent_index = 1 - seat
    if opponent_index >= len(farms):
        return {}

    farm = farms[opponent_index]
    tiles = _get(farm, 'tiles', []) or []
    counts = {}

    for row in tiles:
        for tile in row or []:
            if not isinstance(tile, dict):
                continue

            crop = tile.get('crop')
            if crop in _EDGE_PRODUCTS:
                counts[crop] = counts.get(crop, 0) + 1

            animal = tile.get('animal')
            product = _EDGE_ANIMAL_PRODUCT.get(animal)
            if product:
                counts[product] = counts.get(product, 0) + 1

    return counts


def _edge_pressure_score(obs, order, flow, opponent_supply):
    base = _impact_score(obs, order)
    if not _is_sell(order):
        return base

    item = str(order[1])
    try:
        quantity = max(0, int(order[2]))
    except (TypeError, ValueError):
        quantity = 0

    market = _get(obs, 'market', {}) or {}
    prices = _get(market, 'prices', {}) or {}
    current_inventory = _edge_market_inventory(obs).get(item, 10000)
    quote = float(_get(prices, item, _market_price(item, current_inventory)) or 0)

    flow_pressure = float(flow.get(item, 0.0)) * quote
    topology_pressure = float(opponent_supply.get(item, 0)) * quantity * quote

    return (
        float(base)
        + _EDGE_ALPHA * flow_pressure
        + _EDGE_BETA * topology_pressure
    )


def _edge_reorder_sell_slots(obs, action, flow):
    action = _copy_action(action)
    market = list(action.get('market') or [])
    sell_indices = [i for i, order in enumerate(market) if _is_sell(order)]

    if len(sell_indices) < 2:
        return action

    opponent_supply = _edge_public_supply(obs)
    ranked = sorted(
        (list(market[i]) for i in sell_indices),
        key=lambda order: _edge_pressure_score(obs, order, flow, opponent_supply),
        reverse=True,
    )

    for index, order in zip(sell_indices, ranked):
        market[index] = order

    action['market'] = market
    return action


def _edge_terminal_liquidation(obs, action, step):
    if not _EDGE_TERMINAL or step != len(_ACTIONS) - 1:
        return action

    action = _copy_action(action)
    private = _get(obs, 'private', {}) or {}
    shed = _get(private, 'shed', {}) or {}

    # The production route is already complete here.  Preserve unit actions,
    # but replace terminal spending with SELL orders for every currently
    # sellable shed product.  There is no future production value after this
    # action; final reward is bank money.
    sales = []
    for item in _EDGE_PRODUCTS:
        try:
            quantity = max(0, int(_get(shed, item, 0) or 0))
        except (TypeError, ValueError):
            quantity = 0
        if quantity > 0:
            sales.append(['SELL', item, quantity])

    # At most nine sellable product types exist, below the ten-order cap.
    action['market'] = sales[:10]
    return action


def _edge_agent(obs):
    step = min(max(0, int(_get(obs, 'step', 0) or 0)), len(_ACTIONS) - 1)
    flow = _edge_update_flow(obs)
    action = _EDGE_BASE_AGENT(obs)
    action = _edge_reorder_sell_slots(obs, action, flow)
    action = _edge_terminal_liquidation(obs, action, step)
    return _align_hands(action, obs)


# Fresh final callable: this is what Kaggle's file loader resolves.
def submission_agent(obs):
    return _edge_agent(obs)
"""
SPECS = {'Terminal only': {'alpha': 0.0, 'beta': 0.0, 'terminal': True}, 'Pressure low': {'alpha': 0.18, 'beta': 0.012, 'terminal': True}, 'Pressure medium': {'alpha': 0.35, 'beta': 0.025, 'terminal': True}}


def build_overlay(spec):
    return (OVERLAY
        .replace('__ALPHA__', repr(float(spec['alpha'])))
        .replace('__BETA__', repr(float(spec['beta'])))
        .replace('__TERMINAL__', repr(bool(spec['terminal']))))

CANDIDATE_SOURCES = {
    name: PARENT_SOURCE.rstrip() + '\n\n' + build_overlay(spec)
    for name, spec in SPECS.items()
}

AGENT_DIR = WORK / 'safe_meta_agents'
AGENT_DIR.mkdir(exist_ok=True)
PATHS = {'Parent': PARENT_PATH}

for idx, (name, source) in enumerate(CANDIDATE_SOURCES.items()):
    ast.parse(source)
    compile(source, f'{name}.py', 'exec')
    path = AGENT_DIR / f'candidate_{idx}.py'
    path.write_text(source, encoding='utf-8')
    PATHS[name] = path

print({
    'parent_sha256': EXPECTED_PARENT_SHA256,
    'candidate_hashes': {
        name: hashlib.sha256(src.encode()).hexdigest()
        for name, src in CANDIDATE_SOURCES.items()
    },
})

{'parent_sha256': '62fb5a5f66f0011092a2b51e3192879ba583d5d815d761f176091c615657147a', 'candidate_hashes': {'Terminal only': 'fd75c6710c655f761fb9cd3eaa4e9a192583e08719812711539090bd109ac71b', 'Pressure low': '0a3c8529e0c48f78412ac4f18ddb67f1020086dae955efb0f647f40410de6307', 'Pressure medium': '66c6fb0eee96ae9c7d6bc2c9e10f301702853714f8cbd8cbf992da723fa02742'}}


## 3. Public regression controls

In [3]:
CONTROL_BLOBS = {'Kaito V21': 'H4sIADMTcWoC/+1dW1MbyZJ+16/Q8mLYZXyQBBg7xhPBYM2YGGwcgA9xDqFQyCCPtYMlVhIzZif837eqqy/V3Vl5qaoW2IsfZkTfKuue+dWXmWtra79Mvoyv2je3H64nl+3fRpPlrP3L7R+jz5P2zUxdumt/nM8+t8c3k8Xsatzee769s/2s19lsL8ajZXvr6draWmvy+WY2X7YvZzd3rdbZyf5Bf7h/cHZ4/Pa0/bJ98feTj6P55/H8yYv2xZN3+6enTwab7SefRtOrhb6k//g8mv8xXuq/Lp68Pjzp6yfsXz+//9dw/+3hm/2jJ+rpg+Nz9b9efue033+lr7/pHx2/VT92B4Ovm+1SsT+/Pzx6NVSFn7033yzKv3hy3j89M+W8PT45e/1kUBVJF/Lu5PjV+4MzXc756/6+/tHZQsU8fd3vv9OP1cV5d3jw2/t3RV06FYmK+8VHLPn0TyN0VdLT/tGRLWI3lwuQv2uJXxfxSHVjJqGzwWrtGl9GpPHsL9jy4VKdHr/Pftb7Ibawv+ixWRHPkgWQFBobF08O9pNe4g1MqNFq864kB/zTvBTUJB2eKP39yhTcBIYgV6COsI/yIktTMCvczMBMoLdnxUpjdQyyeIQ2FzSCikLP98/6J64hHruh8pljy2JNp2JdAOeY9eXYSwI+uqF+q4/4xiYX3Efs4cIqrmh6sBPMIIpbQ6hViZZmjssYK4rV6CudI5As1BSJsE92okwKe9qWFt/YrZQ1Az1jyEEc2kzZWirpMLNdBTVKry5Jtgk6JAHnW9rH3K2xF2EM2Y1T9JNUEmnjEJLY7XQPbVLoLwGCiJuEUIQDK162bRxmAqSd+NpTPV6RtP5fqEcHx0dH/YOz4S/9k7PDo8N/51sCJaJwVQkSt2YxuTVyeKeIaTtZbUe2IiFYWFO4CrUmWCEhtCD7VN5RZjG9X50cv3NMdNcQsj5GgAH2eHKsc0X5xeriJwnXEEIWNM4iBwE0HbGdLC+ZV7mgIkKtuLj1A40zjs4Qt6KgGIie4FFTQnmMUAK4q0WtA6TrFYt/s11EmEsRaodbqE0V8FiF768KgSruttkg4Z145Qru9qOCS+m6K1VwHW1Xg3SdzRitLWhRQLgqooprCQOD8X7TtcOxdQqlln34QCnVoLYHY4LFklS7GFGZBSElq+jKNR9llo0Vserpr9dSAJGwrqBGiQwZugh0RwhUdZFWZjQ9S9dFLD+P5oV6kCpBWAlI2SWKIG3aMHU3rHRhEcV6+u2WAOEO1VJ9YQe8ZOgXVbK9Tp+eneyf/9w/OfnXakq/hwKiKMmPmvAj1PvAoF77kdVDvXxOTJgeXCyt4JBoANIltDIvNbgrXTDAkiNowSDCkanD1mawCk0YrK1lhERQhKEiMAU8MrLrbFmg8R1EA181mNKvpWUQpgzYvNKFAdSEUVutmb5kI3fFchhB6wFbs/iu1KzwKCKGUgwp200cCzKLjqvx4yVEGAN4ASuChwGz/1EpjqMUr1Y9jkPfbFZTbgg99lCaV4Ej8/TnpjVpn7JXrFTHQS/YSotbyY6vbnvX2l/TduqhMKvUuw0oegPJhm+YaEEYHNzuYCngrw5/lajGYYg0bWdEqSfIy8AV40Z6l3t2b8nm1qxiHeP7MH+9a+ZD7o1StbilSQ2SJguLOkKgj3GU+1Wo9N2HrdJXHVdLm3znUeN/1PhFGn9R6P0q/Ah83bS+z1H/7kH799BIZCtDOMQeqP2D1WKj7Q8Ed4+m/Refg82Oe9b+favNR9/lht+99idb2/fzeYvmZ/f/TE+NWxiuqK6yrMb172JD5BbV5HmEf3X/H7C+H20Af3r46m0AD6b4/ZHGX++f/JPLrW+kWQiTwFMOxCsSZtP4hN3w45Wj2jWsO6Q/g5QhUlURWqMelQNP74VhtDyXX8YJPNjI1jJ3fHwExwuTcV1wp6raOQC7XVyhz7ri3YMyXOC5As9euE2dm5849gcw3thRU6SenNZPIBQXcKkqk4eBAYciKz4MTinp0CFDznV5Z1jQ8ULdBgn0VoH2C8zk8TBXMemdxfGopligPHCkcYyEMK4abnIIyyIC5IDFcgsLhjw4xtV9nUL0wFOI3dVbILuxNHUsNqGXgbG9EgODAdwTG5DU4gsE+iO6qEJhhhqw0Tz9Bvx2sBgGG8hTgl4LtpdgO4UyWfCYcPKYMmKpMAMm2HYDbRgKSI9sxPBsNT+uSiVuKaxeQyZCQB2hZZh5LBXXUoPUOTQgaMdLtSf0AfB8ABQo2FwFyVGAV48lUtZu5Ektf87jjrxgcyB+o7H9RIQnGFEGJ+48zTud5ZsdbEdqn2r7+1QTpQV3Ny4Ii5/Gt7W4rtyNl8W1O6J4K1Ia3a77vKP3XVgblYj1Nr79UE2OfDMuNoJw/lBRmzeHR7/lbd2QGQLCf7LGhzMieJ3GFDtmSYRahBzagPI4t5OCmKB1QY8EHF2U9T5TQyia1RJv3wV3QndJIZnqK/fQ0iUN/gH2QWHWpEf7b189Gfgjb9bOJ8MfhFMJPQrYCYDRwFo5cACRprbDm048QB6SHI29T0dFlgbvLApG03mUu8xx4iJqSGYMcTzfiG2QMtLOcBEK3+jaRAUIi7IsN6h8slGy8PjkDKcVcBhh6MgqoplTQAyUGwU+P2qosaFBUjQfqBJQY4iZDqIXYcuVDQuisbkqg29T82m/RAfQh5mitu5KsSjKSQ09AA0iSvXixdKAEnRY4wI7lgtq7d4gimsVBL+z8a+YghLDAh62NZw+wBdJXIlAiOAZXp6LoSiAC8KOpBDpnkml8z8QdCEOlGsSs5rPuNn44hMrXTVzqU5gb0azY4heDsJbXDVlH/gJOnyFp650tSCrXeS85t13nWZ6DNR8+l69KKHKxjzwdZ11ge6FnlOOeCSQm8UjRrj3T/h977NVdtxfQm5wNQhgsxUiFIAa4Rcl4lFHJNfiqSeogL/Uyogc/kaO+EHu1ZQ3nsNHj+G9yJtahI8eOFDwQ2sqmEkoxQ4HiMGh4RCJID7G3GqIZoZUfXjVAkcxZzjIxndX2u5QDYiBAre/lNqKwyxECDq4XPJnsF8PFwWgYkY6jtBdw0HMHMLJCUTjUvxbLCJRU5HU+eOBSKdWq0b0ANt8+jJ4mxlSISDuN04veJBCQek8KETH+N9sY/SF6EhIZ4swtKW+NaFgiGv+BcnZ+4ZgkT05I2N1sEj5OgRgutwtvXt+r2koRBQ6MjAszwqGoSdz3tVvjWACjDFXKHz+eAbYFBEckjtxyBoEg4PdkTFxKSS+UNEjdN9QLc9SHH1dCgXumxzYDPwaeYIeS0Uv2hq0Z0hA1lJOXPsWS62EcR5GN1JHzZRxDGwtcdsaNh5xgriTAoh4iPo0MDQOaPjG7R9c/lIjSBTUc3wgB5TYFzvrhh41O9QaCgAEK1GNRkMQBXtU1JYmkhAzkWpnHasBCJx1TNOk7PKsasoPC54RpS6DHFC4sYAKae0uKfUTFJegV7NJsg/tDEIOHuC1s1Rb2MsByKzBSR3mJLWz6czQJo1Z7Vh94AVCQG1D+c3MlJY14iAFtlIO98Sy5yYqBp5nUb1AiQiNKXK15rpwoa2MRqMAgTSwrtBnoNWdqEp5iWCOInBdhqduxdO8564QuP1CSx7AxGV11F48ECpvmz2QFdTZcS+4K6TnuDbane+GidPpfTuYEzwVUNp2Nx61oxffI4oG15zgjQ+JhRo2tMNYM/1NCi0i74DzW3QevN1Mr4JmnG/2GG4I7GC8jjDakNYWe/qtDurqMrgI0LEzwUaHjRH26I3YV7DNACkIFr4NadL0OBNNLS+HRzBaAEQAh8JMuOwK2EmqLwoIJVr9aDSOcteg/NBicqNEvUK4nhG2EIVkVQ2NiDFXoNkOcRzshJS4twHVMTLCvDQAOdjocFNDftlMdhvfEuKtAGCluH1E+RpToEFA+FFoAoDRC/vc8c9AI4NhUAoXgNh4wAqL4gDMmK+Nhd2gon6wIwThwBntNUdGNHVwKwjfa6LOOPOWlYaYcSbjJlGBldpmoiB5MjWcW8OPJIWcRQjSovr2ZJh/LNSTcIBfNNeJ75kr2SLyoO1FvEQ66CoeRFiScRvhrFY5eLGmsND9Dlx2PTPJYExXYf6DXbya3QZoadshiNRD4axtPyKE94oQOl0hfDCoMIiwYbCQCkVbDhLkixtiDJP75LmBgxxCN9iRmyJiZyKuG82sggFEnDSGIzle0TUljDcGJdT1iDyGrX+wGHYkfggUpLtLQN/C6+phpUIaBuqX2UUc2UjpuZ6b/uYr3EAOXRJE1OTGOsiq8hx9XXEqFS4Em6iwmfEEWbhUhVyrMKR7+nAzcfarpQC7lF0GT9PxU5i/g6s9elQzlMLn9uFtzHDBY2KWPAQrKBG0BJKIEMsFFbcCw5jDRJwW/ikLDC4Rppw4Ph/XMIO6kUG1sjWUdNpxZx3BrkW066B0PuC8obwDiTMMjucj4dYfzFXFaVgIMkCyOMXrZ+xFExxvsLCVUUiEaRQ6r0aoHq6E8QIEOMemAyfjjAFhDHHpJoHnGpKt6RI3aa6jtSBRTaNtAmUmxWiHQn9nrzhzMMs5EPLNKkqGQ+clAOHC2IyABfzBARLyy4GaGidK2hGOUX9hlhfpsxD8tNt8ALQdTPi97wZURfto95vAVEOJfUGRJPfuA2m9J1rmakhWjO4kvOEsABOCHwmiYBzEVcLCpAFXEds0IEBZaLSi8GQG1P0o1SQA9sbGM84lwsFdIhJAELzuyXx0Na/bRiK6FGZT1SasP7xOV5QKmdxnupOKDidEdNzGyJO0yPR6XOlvMp0jP0R5WNRHPOAg3/uaNyi88uFJZhhF7hSQWqMkuiNpwtAq53Jn5oe6DE5HhodTI2PSMUBLGVxHhggPTNhOxaADT+YErpiuBpFH5w7F8UAgQmCIO8EMGBlgR/SLGCjCgZviQDPE6XIdfJMTLRJAA2LMZBhbIlAdP8gcNSoix3WVAaeg8EXV+bXkU//dfsQ+iDMpDZxKgRkALwBaCx6h4PFaH8u3zq0VB6SOzPOWLj75+T5j5BbPMuvvNWzLi9BWADPYz4G8DIxWGOega3lUfini40+etz4odiltbmCw7kPmk4pDGwohYUaYrQhd3ltdw5CxHbmpBj2ZpcEWKCc+ooBUBe04jvbALO/GqgXhVhAYC56su6pEci7jc2QZ3SNx4SWsWAno2yT0BeGPFGoD9qTLQo4a/9KdjBbK3wHWiEKnGJREgMG9IixZelRAOd/C1iwzcGKTdcaXP9i8EEV7cMPV0kCgkspA45Xyc6QABz671B9DYSRKJmjP4LGbu+VlAbzCU5sShy8wLCyadGDGnGquZ4KG8RyiYeyITXLQNQB3SeH7vODt0xRqKzitoRLP0MBmExAf3idU8/OCl4LsYt8YeRC7DESrHEoT5S8qy9vt7QsvDghBOKhDrsJ+WUL8E0lAwhJwKeWzC4b0E6JU3JEFVQkCP6kqiUjK4PJMQXJEQo06lhohqQuUtAXOR0Q4tZ97nNmAdMBSf+4469z1r7MP9E3FlMg71MjHGbgVNiLSBvHc0u2RRISpBMR7DFb56IouRAthnf1bcEWncUGH6zVaZWGbPIDhDKmSFPOKCF7dzOAlkBqGvyXd5aA33aoDPBKGkiCIGGWSPQxSGUR0BEFDCSaMtFfMwSoB2YlAXqRLs6V6+PIdtwexInUS+SsZsxE3FyT046A6gSlseNlPOSaaGGXznDpk4gXK0RRc6O+PBC5YxQgfZm+aVzB10C+6pytZBjyoKNA0ciJZotUpx3lObApGMtFQJLDUrqKWtzGrjLLCUELcUSKqs84ntic//0gfmwp109Y5lAj9yZtxFKcfEecC4liLnaqikZRUFJYGR31gnCugoR2sbidjTvLrR0Qpgk5QfJK4ENE2I3YYEc2Ym9kEHHYEShmxW4i0VZ5h1eEdGswynC2a5+iROA+4hpdlyGfdIxIrouv6450A5zBASyJgwLQwUJBtNxaIJrl5hAi/ZYiQ5k024FiNHUE+QEJhcR2ItuGKVOnHMSRtnCbxCthtVEKKkdQkEgEPh4oYATQJ/c/TYZioF6ELgWKLskYL+oQVHYlHF7Ra1sGyI4NNgiEqoY6BcjLbeJifF7d/bE3Y3RywZNHMGk5VNy6vE9qJqfHHItjyETNEOye6RBRDk+vsLgxJRLFzAwME4EAL0cgC10xyLHqsbbjsxOmRxEeZWOKCeTyEI56Eg48HX4D7xhlZmLES1KPRCLuJyL9NYP38VNkr9HblBiYljEQf2gsrKFBQchjHSZdHP0niJtbMF6x+PS6R0RH8AyRb0csam83pQ10q6ibrWCJmMhw+DsxKxF5OqKAqQX27HckPncq8RICghJJPZwiN0IslSNNFTyf6EnTv7bM0kqBuBIhneQmikBBUP3rR8Ji9uCPsRSpYLZwEMReb08l8nizBICz3oiN4o6yDo0Z2BAASEFHcbcC3+Rv3YGbw03zC832DiKMsNOO3xkOU4qmlEcuEItHHmPHSWCEcGXZGgCN3lDCVPo5JeGDHuJ6KNNMWPCSlCQsQCkXD0vEB5jiBKmnIhpnuaeXxGwXh7Nhx5Pz6MS4fC8dsQR4aCEcxIBXGAIk0bvFDeyLvOx8f5FgnMQmbJI3WxYkR4Aoi1UronukMNibxq4WtFqddHNNhnesyzB9BlDno8j3l9QZ+Uu/MqQL3DBwuQJpnKC7fT3QEIGEj8wnRZPLaMBSUd9TCsUkdawCCk/pEAyUUMcE0xWYCgqf4D7BsgEthTMHxHxGgseZ9SvoDxjpV8wv5yT/74a1XhDO5IIMQFTiDmiy8GQCNGL/mda/AVABMIWPXEaNODKcJosVSNEQIjpNGXyFGDhHNlT4chhPDuTmqLKf350KIM5Q2WCYAgmllOKELHwmD3z5hkHYAjp/dGtkG7xHIcwBzEbiDJLGAOABswseYRKd9IvywmU6NZFuhhzJOKaRS/cE7QsM5NxjULUkyFD4iFQOHCky+QXvhwl7uEeIHxZpnhBM0KzCBA3AkOHFoZMzG3NkhSPNcEMaT1Lqi8HfjYsOgLuvA89npl1hadAMDlkJ3oW5lZyVCs02vKBaBLJ5hOB4JbhX56GjS9ZhwxhMk4YKtUDxjRKxQk4Vxxkr4jATvhMZA5XFnv4AsidAQk6Wi4XC7FIxEeYwHOIcThC8qbIKEwYiT3kI9IvkZngk5aOdx90Ak5kfEpD40L96ZoIe/DGCJueN64AoYoczwHGSIxSinFGxvYlgqagRSgYFIy9qRdonll8unYYOJkkQeM7gzCZGEmIa2mDVmEsY5XElyX/VMmbUXHSdEXIgrnsdoZupHnPAx9qAfUPjtxh5k4oQiaFCSuyQSfSbK6D7HA7hQpMXoOKFnJEJGzHs/TO7+KF80BCGILUntcDiSuSJ8Csd9HQC+DEWEI0w2DsI5CQr0uKWfcHAuRFjKqmE5AU+TH8KRsnobwI25/poMWIo2z4UJlUDmD0Ea5ZPGRM65YQEc2DmAGWlS2SEKKxgxBFzFwmgoz1ACryRjEHIgPyppNI3KccxdOkYsCVhwTWOo9dmEY9cO41hqwXwE1dsEatnpMuk9TmiSQlhoviHFAfIGj2lEh51/g7oPdZ3HROUHA4Tr5hpBRDJZnsDguABPM8F2I+EpkDyFpX9nTVKmTzo4FKCwcmADwuFm8Twa5jYzkYQUSyz6gkr8QQWQhd+v+DF7BPIjvKK5yaq9vJJZA2c3HnhWxsm6Apxs7wF5xO48esQ25xFLM5W+56h7LqgFxcwclCfiyC5o4ER1/+SlsyX5McS5Fx2lb8VGOD14CTWZcIUFL8bHQWlCIR1ArTQCSB8cTpLXOlq1cpaXd4xJ5PyY8MxslMpFIV6EGyx5WE4Dw42wSBkoNklioyAcVu7QJjuPoKaRfcMIKeqM4RYd3u2IExpIwHw6lY1/OEeJeybRZcQ5BGMsu9nCOcIRl1jEEA9Dt6AaE1lt3QGqeL5rOIolSJRbRH5nR0CgWKBRAGdHAzN5M5TZHJfH5VAXSG4Mnbjdp/mDg6cJECEaAmU5TPpzbAMAXhcSR+MaBBjvgTSCwB0r1UkMn9+YI8oBTWcTBKSW+WTKwW0RXhjJThfLxCClpILAllVfPMmS3Ded6Rss4zzjwCucgxcAUT2zlBBTkgUKPovOqOtWaHNJmhsXSriL0O+69xVwLwYNb/uRhvdw8nuI4vd9Y5H3ABwx+s2OyHFypYmE+VlBqOngsBcxFl/TmUAkUCnFQmOwl/zjlohiDFEsOx6MzPHR83fhPUdTgkB0JVpQcFRSjoONJTcGNz5BHSDz2RlkC+otNGiKvz3jyOciwpodVg9oApA+YL7Z+5hn5DkiQSuRsIsuUGuZ96cUPMOVZtL2zCvMjrDICD8RykMGgyOhLuuC1MAsf09oSAYFVu94xdsS2DTO1YKMZ8U1hmLX34cwRMV1dawwDD5gjOj5ICBkdQYJdLnkpEL1Ua6jrJwrolwlXXmiAKI/OZxNCtSMnJClG5QrgBNilAjSyVnP2OtTV5ibhR/HzencjEcvFIQpYS5BvRi9yQqoB1Wt8iIBN1rfrYWTc2KHMPwDel+yun1bviw7HA3cJ1QgXAcrIUj+3aDULd0Qz3EqdiQRhBAe2tLqSXsRXIwR1YdNdeYn6PHrQQfD1YmVSxdw72wn2dQDQyE+ZyOvSIKUXMfouGe3cwSsGKmVJTbrfD/4LZHh7LvAb+HVS8gU7Xzz+K3DufNe8zaLIvZRdtN538OSju9Tzc3g7EzbTAMs95LE+RwnTPFz0DB93xtM2yyhrTKCFUD0RkgJjEh5hPoKT89KYQ0wnYHnAR03cbML7YcO+HH8lRVPwbMHaLyLw9TD3VIhjIGmM8rioIcl/YDkgs5CSASEMiojkk2hejiGN0EqocIKMoizHsmPpcgkQdN0JcbAnBFZHrA4oy02NMkdnUxWKqF2OFQoHAaNnU6ZdH/DAya4uluA1FIbQbOd6mb+uqEBQSgMIvQhrzcRACBqXmlJTDnOQRKI59WaXkCcCwU32UMB/gkchbodkmDgjk0bbBLlYuwpREhC17LsQDNj5JqWA5ZUkMG8OxnxAMjYhM1AligVhox0SVQE3mYDqhej/5jJoFlph3C83bP3oqKx29FzT6dO57CU224xn8G4K7HCPhxf+96jr/29Zp8WfeQhpaxpLvf06uJX3kMiaiLcG8lA7Aui3vna7VErTHgth0a7XF02HBE72TOIuQOhDkZlJYuZf45tSV+SBx5BAKGE4c13caVY3ZATKrnahFlhUJPD+QWIbMD0nHNShyNEgJD4bYMZdNGgLCyeJj6nm/LjhFwKGQkeCDPTMw064UC2A3EJenJvXNDokKmErHzjmNdL3P4858anFTpW02OZdYQK87CABZ9Ny+aH9IQqH3edEIVnYCDNImg2YqoYwXEcTMQWhCEF6xIjEGY/OIIko/nRMKWEnd+DFrEY0DiDf8tZqlDCDAZC4gEH3as1CHxSsRMYXQpG5CB2MFaMRwq9JBQdARBGUYljiOum3lPtRgQvQ8JeekjMSUFPnkSyI0GcM0c1F0t81lSEzee073w68/Ye09M8+sU/MF5l73vlVVInIM3mT2iIWElnzmsyW00wsxIPm9kwPY/RJZy0x/2gWKesU/wwV3g4eClg6ZFUFXauXdZQw3ctz9CzkizPQr5eQKjano97Lk4xtEwcysdaFErTF5tkJ5mt4wrQkkGFoKWiasRbOtjZWiHnfKIaQu5LcMxSdpJjLBMMw85zc5eIA5NVUC2hISjKt803BtEcZLHrCq2ngpTb5CIhIi/F6dGumDxLgFjgxCXrQLBiYntGg1g5cbrJlZVjaHtwK3sxnL8dECVEd2REp6SDdrL9v2Eb3rPqOA+NyXRkcAEIT5IYLu/M2jn6tagfCWp6RCEW1S8KKTgbpy53LX6GIfx0Bcy85RlDcwdGoCJSpcHk1UD9QC4hmZUAGk3gwMDPAdj0aHmTdJgJvd1KFuUSf45mZyveroWBYJ8QIK7zO4J0Qn6czzSzdzxeCw/F3aFDYIDbAsBXFa2pne59+9pTDf69YMK9VccUeDixUnl667MVj4P4OLEo7TWF70UASZ6FDRs+Lgy6S7OdkBsMmMqmpYYk2YieaqoziIUoU1l6oI4TcfOqKFTcAJY4QYjryR3Uo/iBRihrCwLx+JECyYPvJvmSjigLdEQFEBShMBUywQeJqccdmdDYC8b3uNdCpxrOBxVF8D5ns0RZwAmSl4jhEgFFWiXAuJCk20x/vdUc2uAHA7DUkBcl2Hf4kAjTfAfSFC7nXA4iAd7QFEXBaOwIXQWdkVShg1DXWHQhmEC3UpnLY7j5sqPi9pkwsDgvEojhSfFztCt3pIRgv2Tr5/x0StLq7Qi7ECRnguAOyf6ln+VO1Nh9CI5TxN+YOWKpkY7hsbH7kaiim10JR74pB4fNVhwy4j0bI8WxPT9oL+6gsMMRQMFxiXMvSonio6YxJzZ4SgJOBI9jPU+KbVivPxP2ehQ8t8uAbleYtSqSo/72o6P+t+Oo/43lqmrYU18SgO2bd9YXpVwKz5F07776VHIe/+h7gXniHa3P4n/yWWwuXStiSFXPZGvgZo93oWPK3nvieAK/DMCdQfWJPypiZl3vijjB/JAhtAbo7+/uNVoJf2PXIBQFkqCRC+988lC/FDaITEzaVic8Ywk2x15UDi0RXFaOqvPZaixNm+BCOhL5UWseIW7c4w42A90Zl4MAXkVRkzi5u2IlZWa4gJ9zMwORB1ZNuDsQc+KcHfWfOrKvAz5NVYTJ+WelOQKpjNhxI7m0cU9FCSgMy5jo4i2CG20lzCnlpgbmOOOen0AHSbig7JgVwpyrjrg1FLQeSVgnJCWkOEBCsMIGQezLmPUgKZd8lLUjjiTqCEtLjcjiNYiRihxfUDE6vVmlDRFH9/i50x49/b8DT39r0YrE6mQAj4I87SuFFR0wYQVA9nPypVxfGnbgJ3Fxv3S6DDWcR9yMiZPRWCCcnlrigEx584ZmFZF0qQx/lgYOoANSkNTsmECvJJIqAf6icJFLJW0a5CUgQtL2x9P1uHsulAQoGbA4q5sKqSXIjNt8pViAHmtfYcAfuI/+eb/RYMHw2kdHNCXADsZ8xo/+iYpRoTt5jqbiXBmo719YzAao76FFjQbGiZTN3sKCcuH+sDjdkCKURoCNuEFQGTm+cVjsvM880XDZCvycEkx0rDBf7c2ntAbRRDooQ2FwHnrEb1Ga8AfqX5CdxIepqPCpUdgT4gALRJRKKpkp+DoCQwf4IOGEMSSMpGzDlaSeq3e0TyBiMtV5nxlhR8CP48CegcLyw3OCB73V8eSB0VazposiieLRCDxwTdq//D4gvO6WgCsYB8KLTi7YuQ/G3LcK4cUC7lzXXXHE3WpOGIQXsQlo40IGjIhy1RCBESNWE9K3IUqKRE0Vhk6XTEDveIVMgMNRzdgBLLthyxBlEkBUHFclcRofGRiqYe94wgOZNknpWHtBPUEPDJaeSeb/Qe2lVdPTqEMIIkCV+5cHXoDy2UQp2oWLTBMJLiSzwHcfovEn0DoTg2MgLi3xxoMMyyidEYdoBoknjafo7aHNHE9UQlO2oyAFlksir/lmOWUTNCtsLPQkhGDjFIhG37mjRsYNqcy6AMFJgD57BknhOnGSLsKSxLuIA2Jjzc/wPeUfAIC3yQo4ZoY04rmNzDKYW7K0XDXXU5bvbCATC6CnUVEkiTTXEO2OZ7JAsR3BMJco49s52sDhxG50/nLLqhgVmZXIlQ3vHRBTkEBJkGqFIGf1TYRHd9uFsDIkNzYccbRHebIi4IaD4UsaE7C25CUCqAUyBRAcflCycpsGd8WgaZ2Ud1HAubJbYXRp7OzYwGIBCXWYOHkQMZ+FJqCjX1mKN1SoxCDxPyexgjL0+eF/IPIGtLNQAZ+zowaerLxzMTRaObRTg0dFvOjjzGDbxIkskB4QuYwHZSbGABQPG5QOCUzsjHuBuF0QIwDesZiRq6FiwUaKJX4EqaBrD1Io6Frxi3Se5jp94MLivhtsYb3is1UiawslxzuSK3n1uWZaGfcGwa+532XJ+jzK9G/mV9A0exTpUaRHkQJFGrTUvwO1mpy2X7b/brXVv7XESF170f577eNkvliqX92vm+aW2pROjh33zo7f7J8dl+7tZfeKFal0v7OVPfCmf3T8tn7va2v/7eGb/aMMQCqEPDg+Vw+t6fVsLSvkdb//Tl/Umk928dfj49O+vtj/9dc1/cH0S8Of90/79Tp3d6o17e1U67e7BdWq092q1KW7k13RZb9o538mQqs38g8lAqsXtrILxT6atMSWFvz09fG7rB2s3vp5/7d+IsC6qWJWl430U+8O//3v/aF+OXnGNFhem9rjP5+8f3vwenj6Lql85ZubpTpnr/xr/+Tt8PTs+KSfvGFaP7t5eNAfHpz0998UIlif2GznAlWk7p8ND/Z/MV9M+yL/5umb4+Oz14d99JPZw7/sn7zpn5wO3+yf/NY3dcork33Ybo5yBb+2zJ3h2f7Jr0qm45NX/RPV+OvJx9e7eztKw06LWu/1ehpsyv7s7u1utnfzm1td+2ZvSz9bvLq1U7q782yzvZf/9Ww3fXYjE+f0qN//px6/u63X/ZNXwxOliieCnc1vx62T/rv9w5Ph4dt/7h8dvsquvj/tDw9fHfWHR8cHakqdH5/8lt3S6sHQfDq7pD735vCteu70XE0rdfWX0fVinM3Hg3196XmrNTxSapPq/eSRHzqt9CvDn4/f//paz9cttcBcjT+2h7+Pl+uzD/+92f5jfLfZVpdGt9fLl29n0/HGi6Sek4/tyWIyXSxH08uxefRqcrlM7+p/8/Hydj5tq1tP9dfsD220rAfUzdFyOa8Xt5EJc3M9uhvrBxbp59M3J9PleirpQg0G85gaFlsb7dlc/Td7Xy/j1tv6z4WqrPVqckm9eTFIXr0YJA+aD+onLQlsAZLXLszNgW6S9I0f29fj6Xpyd6M9Vj3R/vtrXpn55M/RclyvjV0T84wS6O+viUCl1y/HC+tts0GUq2OulV6vFmQeMWWpD0JFLT6NrxxilmqhPqIfBT8xHl8tuN/Qz0IfmUz/HE+Xs/lkzP2U9UapU9MPLifX6lO6e4Bv6cvqE8kz0Ms3s8VkOZlNF5URpXsgH2hm+GRPqlsX15NFOljTAoxuoUvYUiM2LcX8HJRffzr+shxPr9bzmWU+lt3faH9Ur2Z/qUnRLpWT6Ct2RdLPlwZyXla1ze/MiBpdqt/ZzC8aV9e52jvZ6pC8ks4E65n6AmHdvEheGtSHbDEYdL/UZbKbutJDyf2kN/U90/VFR1XE/ellIm/+hbq0eglMrn3ZbN+pL+aPprJn37tTVd/SFbnLPpoUjXxwPvtLfS956uIu/86X7Dtfsu+o57CvmL/VQxdf8jE7WuaTOWq7fZiN5lfDxeR/x227jlpu82ay9nW2koc/ja4/qsesd/7xj3Y3uTW6VEvQIleTkn00efyHdmeznf1Kd9j8rvtO8R78Tnr1q7Tzza5qXVje3qjhWB0DG3oSmjoVa+nsZri4vbmZzZfWwpFe0TWfLMefX6iu1pNZ/9bfsNVfI+xy9te0vNbrK7WVXn9EF1msBvox9fTt9Hp2+cf4KhGovCwUtbVFKCmyyUau39xsr29YL1hVudAvDtr/9bLdsRsqvZsvL8vxvL4M63L1LFDF1m9mD+gb+gn1YFmAskaiH0tVEl2/T6NFomSYy2t/TKZXa5UK6H93k/H1VVJEJunl7Ha6HKrGG46mk8+q/63eS+4lwza1cLY2C8NG/87sma2iVzLx7SawplUhkiku6+tUbnNRdVuiibWsqqePqy8bqcp1M9cuzENp75gBqJYFXUa+1+eCuj9Y/9jn0Zd1VV+tj+kPJcPE3C6UsY1W/ul8czENAW7vtBhuUWrdmsq2XhScTJUNUNDS26Vd0pSWjYyZGtFT9TkzRBZxFcz842qgjr+oJzpqTTMPVYZeLm4+BosrxVgsruVj0n7Oto3tG7kZbV/MTOatyipakTldTo0WXFtK07bEpoXRrstfHXjOkMu5WgzLT+pLcWaS9XJSDv6qfqS+QJbH1qex2iP/HF3fZrpOOkavRnfZzp2o7cnQKSyCdEqrld3M6cqeY8pTmnLSaND4bbXKjfDyZTqsijmpEZ9kOo3nf2qRbuazq9tLvYXtbba7mUGfPK93f9abymLubabISittlc+jyXQy/b2dry7d52oKqAZQ/00+ZapzeXdpVJQtLXbxmtaajPbR/i/rslI5MgGKZlRvf7yejZRKnTRlsiqkwm2W9uCL9OrAXiqKPTxp+fLb6TOXs88346VR0F+aLqisPS17MBQrmKle/ud/GoGtv9c7T7dUDbeedjrqr1SWYjz/o3hga2dHPWFJspHaAemYm89ul+N0lxvO5lfj+WJd6TPqwc12MggXy/FNYf1PZ8u2jWZojUM9oBq+87xbne6mduajWlXU9ov5dtIMhcFaqCPlZc6xDycP6THx0hSuNcrtVJtWtdFbm7bcslUmEUBPTyPJC3vm6vqUdw4b39DPbyZyl/cHZVwlq1zywIZe83q1+8m9i61BMps0wc7AMmuOBzuDRGtL5l2uTxSlVrQWU8+no5sbbSIaMSqrznQ5md6OW/nV/7kdqUvLu3Zp4zaFdwfZbp0Ldq00xlE6ckFF52urtGMPE+1sNP19vJ6VtFFdCP8yi5vu2OpSl9Zbr3Rl/fLTeHyDvGbESV+sKobp0pt8e6B7qUDG6krg5aeZWRTSb5YeGF9bnzP3BR/U5UOfy9rjp5d2PYVfWowZdQF3pOShymZW7vvyMw4NrTpma/q59b1sA/2pvVWXujyoa7cTDMSeScUGWS9hUHs9HR5m/bnI1p6BXpiTci9edLYKrGj2ebScDZejuVqoFutlwGih1i4Acr54UQJ+B8USO1Yq3OU4gcWqS6y1uv5+PfugWrSMzuZbcxlU/vGlgQfStbe7uw2vvTlMyF575+O/5pPlcjx1LaLmxftaRE/7/VfIEqofs9Ta2oMV6PvHcrM6V9usTRpccNMRZ780mZYbNbu3WbpaHhg/VKpYPFsUtVjOR399GM/nd3Zx+c8fqrK07L6uylmbybXGasGTOOlJ+1in8uUBYJFl/yrdqFanqsiWxFB1g6UunWYBJVSld649Wbml5SdbMdJaqT+0qg6sGekClWB1lSXL1tjSi/AaoZ5Z159MbGE1wWsPG8xalWGvHwWQvaaPrdcGJWvK3L140U2mpHrkaP9t9WxyUO6BrHHSD+vGsd5LR0m6ICXoNrSq1WHvfAHTt4w1uWl+m/d0jcfTW1WkPklI3reNzQQifGm9rJTqkulXHihpQxqw1WrM2kJkSUC0kmtRSkS9KARDGyxrXNM+g7RGueGZqv8f1T76YXT5R9kAKJ0CgBaAvmReqRn85nKrdIRwZx8glI4aEHAqMeTTamwNipGbF6gfeKnP0rXj7FqrusuYV5Ntplu6mX4T0b9LMMb/3I4XxsTIXyxEWH5K5kn6tjFN8xdS9c3Yp2VNrjaOSpha/o0CrUoUgNqgKr2VCGO98ZP1Qm1/W9xeV7SEDeCJZIc1tazcNacPyUNI1xztH/S/h57Jx668e8qv3m8fVaaoWQo+z/4cf9b4kLJFJvbZkVoNsl1HP1MGIRP+qyZuKC3nB/tgZi0hIKd3Sjc0l1Nf7+j6W9c1MVxf/8G68XXFx33FiQ923nelbl4lmppqj4vZjbk6VVen+qreKq6S99X/7/LlMz8inFJnhIUU+r38SHD6pfTexfRuwDmzyp69mKqt4j/UkD86PvhNK9NZx4/Uc6lZrR/OTjsYEGqGgOp9QE3VhKtWkyhhrKTw91TNlqEBcHI2R/px6+5aBZ2vvGAAAOC5tEADGtqlqWZLZDNg7EXKY8u1rmTEp5tyaeCns620BTr2O9Pu0I6Vw+VmLdQ6k7uNzIPJsmem1mY2kzbTmbOZzpR61xNTuJXPlWyq1HZg7katqRvJTp3zPVqVaiZ+hfV6lpdUu/uTI7p0bGWbhlJoamup7oIPs9l1aez8pVS4uerr5cyMiqQzLBtroyJd6jZCyVcdoslx4fB2Olku1hyLd25v1maVU5icT9hEc9W3Hpu/6NqBahKq5UIgXPlQpj1ZJL2WcxbI7vyIdCVSsYw6yKqTduFYZZ0uR5IBWvfZaEhWoO3n6tfkf8fz4ejP0eR69OF6zBDYDD9CRqfSl0ljjgUXsKiJkqsXHHOUkql6jP5OltOf3x8evRqq5ffsver5zXZ64eD4+B2wltYkKar66vDXNefzWUObRQDtE+DTqQ0T2owQAQhszsLUErdp0hInx4CwUOlGrZ/e1ddXA4TDC6m23s39ScEbu3uaXMqADsdgPLAnt7pjt9mP7e4LSEUuVKfkJc2IgUwKyTm0XqYrT9orty1hxsDBUXVwQFhSQZMn63EjjNFBkkkAPlVfWLVkyHZXr4UuI+UdxBc+mbCxJYeXVO40YhdaPvZNlL5UBdXHGNfDv2bzP+qsvRgqG0eFzmYRQ894gdvJDShN4Ei6yDW5QfVYgqsjOr5r9NeBD6Gk3BwZQ2QddmPYzPg1G3FUt9LqDm3ruivUr9TMUia7/oj+3496jJDtXSirA3C8uFUPZMisTgl01kopuKUhxB/FkoEp1LYcnwaUQ3r4g/qnqwCtG1v9y9rQsc2cs5HXdbALo2AMSotmdupgnZiMJvPUeC8xWMpm+3rF70ZJBPjcbMAnJeqrt2p5Tqq+SLnPFSBqoyU9McmfraHw5k6KwlsLc9HY5cq0qtYHgGhsbaYFpq2tG6D4HtAUpY+mxzrpeWxSi2qnFdWp7mVblVMw66DH/PQ725kqWyBBfdND1nInaZp43nC6v8wJj1rmzHuF4MmNzOniIuujgSZbpUX8YH0g7eiEyGF99if4s9l5y8ULc9s6V7dOlnICjVUM+yjKPtICxhJ+dFQ+Uda9VT2am82hk+TaEu4YkMSgTMEpqwrWjl8eodQohc7XasPVcZpmNR+sh7WwszfrC7wzt+ScOTu2XYyUUkfw7mwPQNUodUZILA5IOqbN7cS47NS/nVNG7cWfoH3UKR/lE58S3UNXd63lpHmkelRpc6mQVOANBaPAw5yN/EytUN4KfnvaDvmb7o5wk3czPqouzGKjFsXtbsHc0zrbtyanWWkm09H10KJMqqbffb5r7IpP87H6zLVuju52wphIv/9TxqbtbedraPGl0rzMPrq9s1vqsoJxYdN79B1TZ3U5L7/aXaYTQULGhRkdNn0kL6kkZCK+ZtLsFU9sDFo0McP8LJPCxvPPyYcXf43HN9RsrTjnFmQt1e6xJur4i3rS0KWLw7hslpAzA7bbmmBulafyhnWmB09DeKa0CmrjwrDTbLelzfaH0WJc9Z96qm/lBD7GFC+b7aWtrjTF25a3VNYNL2BGWHbBMcsL4auuKGldsxlgXlIKSU4ESyueD+xyGz1d6EVhPv5zPF+MX2qYoWDDDKvvJu4L5r0aYubYCBJfwPl49IdjxuaTtFzSQDLpRr+PpylOodrz4+R32xs9Y0zm/u2bIHsymXcVPl0FAtGPWKt6oevo6isz/aA/3D84Ozx+e7qReBm2rMUjoZiNE8jR4mS+tMQqmgxwvU8u2x76+n27hdSly9nN3dMrtejoH2V5LvTjWZMy2fzpsxQttfwYwkXLH3SZYOYBlsZjHiWXWojB8H+AurJ6ElgCAA==', 'Syouya V23': '...', 'Zero-Waste V24': 'H4sIADMTcWoC/+1da1McR7L9zq+YS8QNwQrrMg8QIowjWDSWiMVCgdASu8QEMYKRPGvEsDNgCzv832+/u7o7K09mVTVCsvTBHrqru7PemadOZi4vL/84/TS56Fzfvrucnnd+n8xnnd/Gi5tJ53oWXbjrvJ/PPnYm19PF7GLS2Xq2MXi20eutdRaT8U2n+2R5eXlp+vF6Nr/pnM+u75aWjo9294Znu3vH+4ev3nR2Oqd/PHo/nn+czB9td04fvd598+bRaK3z6Ofx1cUivhT/8XE8/2VyE/91+ujl/tEwLmH++vvbf53tvtr/affgUVR67/Ak+l+3uPNmOHweX/9peHD4Kvqx2bhz8nK4e1zceTM8ODCvdkejP9c6FTH//nb/4PlZJOzx21SGUt7TRyfDN8epXK8Oj45fPhrVqxB/+vXR4fO3e8fNr2trlT+90ZTy9f7eP96+rj5sCpqJZ0ga/0zFF8rcYyRtynMQdX1eoiZLo0XDyvTm5XD42iJV0UrmC0zZ3hy+zQUhxKxdEgrZNzq7IdKPcd/WhKh3Zq/eRsPdfNjRrdUc1nkzEeJ1pTOhFKts4qrge7tEh+bzPP8lniINAYqXWkZ2NOheHZfTnxZE+Pmu8PP1gR51ye7x8Ej4cVVH9azLHDcB076ydlS5ghlDP4CwXdlQNz/K955lgVW2YEOovCXo+VdterpnsyogwXpKwfKOsYz1st/KX6oB3iOnfZ8cSa+MJ2si5Z1K9x/Tp7i9mk2St79lBBn9Y/7UbCh9WUdQrV//Vf2islmby4q0eevv9ag41dxlf1taO/2JelfYztQ6hTY/SwUH2pFtzCuimUspUIPXqz4QtrQxnsrmbYpsbXJ5O5AbG7kxwGklqCwc+I0KUteojUM34sX9QI7zZqOg+Q50T7HUVctBoVZCY4XesuC3sFZdTuO9w4OD4d7x2Y/Do+P9g/1/V0eT3pDqu4hDbqOchaTSP/zaB7aUXDi/lhF8vxS2XA1JbcilSSyfr6/+KmNDbVE8Pzp87fy1cnAYlSAVG3K2Got+8Rl6T3AzsLrqjYCUg7jrK4a4OZjd0GEwFEs6rH19FwxbfVoOqlrmWuBcbV6/9aq1e88GqVm+VtgMTqpq5H3fHm1CnZZqGV8XCyL6mqGutv8xVDX3HpVWzfgCdfsLqVmwser+tbL1grYj9bIQn1Lq2E1rxcREart2e3q3StW3Y4AVPOfw8MBift2Htq4Cv7p+xkVIbV4Oz/0FlHqEcQbU6smOk0KcAjm6EvyOw7P8LAoeSkJjNYRmxUGjAmBHW03pr3Slb81SYPEal94FJoJhnFKNDE9YwtkIpSS0Jl3fYh1sBKM2ZL3J80rnsUx/jRq/yo9RRkL5CgMAoL7Af8uiSmjPzcRKr32gfTEKNrBTwurXZef6GUXWfhZJUX93q6bZZ//YZ9TyiUn4QPR5UsLBw0LavwQtnTrRHvzVdXfZaXQAQB6o7q42hF13Jzdn+ielaLrD3/ehsQPgNbRdwkL7jhQTB4UdHTp/rfo6Qr61FQcKOzm4WlPYwfmVtm68xg50VKSx35NarmxgL7W8pW+5qOUWVWdr9E1f/6avC/R1GrB/AFr8Fwaz0+ZHfpQw+AbDQ+bxPery4a0KXf3dKa5OOPzL3aN/Cv6Qse7EODW9qIr8UuoncX2r405ftjTZK12yb+WkTJFuyNIu6bOKUhbjYiEt11Bvjo92T/4+PDr6F+nVoCTFWzTJXHjjY3arhirM1YBTC1l6Kz2tKOPDYbADGnHljeb4Gb5hiLaiIQXNG+uLm1XVVpqn5/MyUNeoZnJpeGr6QKtWfeZKGUPoHIg0e0UICWBUMS+mfoX6WJCqAUpM2KrxNhbFOiXVfz/WqYNRyVwMb1WKqxycB9TvcrD8AyMCSbxxH8i5ASe8l9vuPRgf/baND0oxadPpQGeOEN7Q7iPCqyXLVcMmNu/EHcqIMtZmwvmehpnlbviBBDI9tTGtW7umK+w75pzE7xjFgYkUpPLQcGM9w1uruTH2gTd+4Lo3XIDsqoMrr4+0MkhDxrPCohWWMjwqXqE1/3sHyr9DN5RWedn9ah55qFM1pMwFPGADB4tAJo/GIA0w/oiabAsnKlkvAJUMCSY5CwjuAvBgPtq6swN16HLPbg+bD8rt4WmoAxZbGA8xsi6K6ON3BuQNxGstSU9DAVoxpNIRSuUmIWedHeNuUYGALBXdj2wFIF3YvY9sqhNCQHUgA6R2kuCrwFTzPgvkFyHIMlLizkAJ4s5n8FGapvfBmQh9miWAI0StwLc5agQ+lIsS6ifdAuxUL48GB1FCEC2xDu9a16LUh4/dndWzkzQ8GdSXaiYB4kiKwbOjSJvNElwrWBgtRISrC+qwb/ABJF0cinWNQjNnNh2GDglzITaSMDyadJ65OLJr20u21gNJ7Fq/Q1OIBeDpWr7udqQYFCvPRQwF7csBQRZ9ra/nT7VqCm2yZycHu6+eBzWMEI4t1J033FhotHt7hSsycAuJ637opThX0vXsRjBvGlqpI0xcrgIBLTMJtU1/AiaXFWz4yFuGakPnXlbbarhl3EuIY3Gqo2SRRpbFj4Ep2prlx9P9wKjzsQaFFB+7Z4+NxOa7kcuDy1IQuuyiZ8ACqi9KVQMIIyL0SU0qHuEXHvO5tFnA0G3gMIi0BR2bk7c7nkpFp1eSodDnJWDLigmGyJYrRy8VvFQ9UsWqsRwgEhOy/MPQCuWkTTvpiQocBYLWA75tLlINpaGDPS0zRF8Ex1HyPvaNDC+XnnF4Qie6oYXm3RAdRFaENH7Isrc0RnwPObfsMGrfx2B1AwBCYgECw7kXEiaw0PU1KWaCQgZeln+5KbuexarggMLVbv/gH8WgvE+MwOJ1YTvIFNaYPyWokhAZdMcPeCAsd6RoyfikTMzpNj0X6YNbi4zkoVN5kWfy1uEiMGq7vcBrN92NGj6rI7pElQ0ylp0agZSQ4D0DZjHrWKViuIgZ+ryNVKVG2OGpBmGSH7Wk0YoVEWl/Km2COveX8XczhjiwXW0D38jCAr0tlRaX3XGSCaxTrZ/EV4CEvxyrAaAkcmpQMqJokZSGrHd47WoRJrxlA7o7MbOaA6sntkgFPSIm5VMTRaKLGMwiaoG0z3yB8FRvkPuTGHm0Yr3GxQZhK6D0AtVWUAO4WBFwgqAuaHJQs5vVue1zmV57KpBHY6Yw5AFpDRSHA6iNEdWhESDAk8xn+Z5gkMsRZ3dHZ4TDWVZ0ECeVGAR2DNlnFCPxAUOLulZfDV26n1SmBOuDK6Nfmiiv+j0EXYHIvULsU3bwZkGVgmNY3aeBQSBqVIY4924DsrHZDbZknX4ATf6yZ/eMTJncQ0jTsV0n0E3yvcR5Ughyy7PwMB325bA6JLjEemqdLyPoZiggrrsBbe1areb7qAvpjQDACxU7Ky3s70JNmd7V2W6zTp7vv7DjkB6kKClE5YrEdSWoI9l/1k1SUCuQxs85PozRgbQHDKLknDQ7tJBdYdy4k8HIWYGjk5HJ2QRqOq+M+TNQqP6gawNAJdLkr3WNOI5XiO6BVhKlldYE1sHWvgQLMGEA5Qp5U7s5KjYSVvnnlDdbm/cXFPSQzIVJ5iWj1c4RbgA1BKp+9KpiybfoMZ8GHsudIgw4rI6aygloSIguSoleLlgsNI45K8FYazzBk17PHJN3CQYRo6aUcwduK4iUh9YqVk3wQ6fo8YgwKQBmNHsp8NYgb09IZAa+WiL3T7B9Cb3ABAigS8J7GEm8RhPW4a18dobA7eyNUjHR8zZCsKlCIFS9rwah6va+RIiqeh0AFe7dvN4md0xEkHMJlKHkmgXCcbTjl1TdcHUt9EGK9wNM7UAx8mxQVfkLU8ROIJPuHkGsoZAoJHaAU/GsXLuqKxyOVAdhDzpVx5Fnl2oUmNepFMYb8q9TMledFVqAB/KhOGjtWma3thADWxcq3japqalWAxBCxFEha0AFiEf2pKTJPXxCgVmGQtHYAtTTzzXMVT+7HpkHgvmmQmmB+6u7v6QE9YX2KP2T55/5cQ0AJAG5PSTX2cWSy9N4bOrTTxCrARXLHfA6YI8AD5uyApxd0GUCiNeByjKziTIKqrEKoMyGPEkDoSaqw4ueJna6PJ5AlQFMmOzV80IibAoGVwL0fF8ZogiuFbIwqtVflabggEeqSy2BT+RhChyDGiGwiGah69jBfTt28mzkwo0SaDcg2CsfTh24Ojt4kXILMAi9Rp4xWsLaiMeqcKhLUkP1wnHC8ld2t1r3XgyBt/W/WLztiwbWyFem06AXril6nwNlc6d+QUVY+hO7tbZODaPjJzZikrgEpPWih5CxKZzcIFlOlgiz22U9xmSxtCAK3/fz+AMAnQX4USRFaZdq1vOrvQbJg7GxtW5cAQFXyEbAHly8m2djRLj3D2KXkLpUeZEKJwZOJylAmScIOmAiAEEF4Y5hRCfgHKHyQUM+I8CLjmQ+8PQTEA7byoIS9YY8zhCGLlEaG2rlC8L0IROc8qGlBSur6CTItjYAZaNiI27KNHqBDw/pRYVUaN7vGSSUDQ08U9CbghQM+WYyRuOmmvYHpj1KtCRhaLtRoSy+zsHcykmTGwK/DMLgRJiyrM5WWIa9pV0kEQdPPgwAsh7ArRW4NCqHpGSAuPl7yo6H5MnF5Pg1kpeFGRUZtINUBiZwU3kTMKFW/aiUICUu1P5JlJI5xPGZIUGCkus5TDyG19CnpDb94P4gPBVbA6AkAsv9fvlzLpXwxP4eFLSnrQc1dEUKNcJ4nLC9ypzfDBbXS5XVVh6vPEDgQc94Xjw2V16TkSalJLo2Qsfb7DPQ8MjNFS51bIghmecNVTPSFMLwG9V33pxBkWLAc6QQZw9PMLEHHoAq3BOK53VQxu4hm5qabTyOYWG2+WQbQqHV6crJ4S7k1gbIOYzLolcqMuDNQ3WNOJoX0k/r6k1vXXiMDnzcGHKD5NSCmUSgqY1hiS1NKw2T8qXSHYVR48+6NNciZvZVs4KygCTWl8Qh3wDwAW0HoC0WOWDAWw+3rgrKZ/OjpjOry61zfUC/fpD42iiRAGwSuX0OJlw+QWAwdiW8SHqxaWETGfWN8ZESwHvFU5uyDNoIi/LuWk2DuECYPVk9rf1NZrbwHATUyDbODcBoyJdd9wzoclhZnoqE7t209cSBEcQB3vmRSqOYVIhOoseotYzJGiD31vMbh9CF16XTJDFG7FXNuzFSio4OHTlBSMXjASwEH4aNGeeKLn4l4KIqItpfBkUcCIZgw275XNCic8A4V2TRmv4hEKFMEMKDNBDkdAs+jzeVi0V26M2jiDrzDTHoFNw/PfTRFZ4E68Ye2Reg/7SLosrLV4N18Wgeyb0BPBRAaSIbToutdmWgHY9VkZgWqSVS866mj7huuE5oHC1Ps666MwzE3QRvc6+wJlY6j8UhKifgZTiDraIqwYyj7C8ZkcSDLN6rrJuZ6TbQxihSRCOS48d0YEjWXsvk3wwQyA4iAEysOn4QgsbQTixKf6OBmC0hV0XqAytIrAJgeRSI3nc9QX7l5H2dPiPFj8ImWJaAwkUMVWrLg1yx1vvrRHyuQLLdnUKhwdiGuqNeB1iz75N61Y2NB3hhiB+o9uuu+10SCy5AuRVeJKjHS3S0nA7SYASiMPQDTb0BLGjS43Jp5VVV5VJQhYhU5ozc8kUSifkO1gC0vSoiIAhTqG7pBoULYKzIUgNcB4RZYByU4IE/ohoKlISJ4ZzJl+sKCK1OH/XCkVTx68Kgpf374xzqMCQP7LS527XR88MXL1wZe+4hGy0D3CWPbEvZNqzwr5snFg5cqPKsPvEEoNzrjwiyPHeT3AVIRD1MXEMuWR7nZmv/Q5Ns5H6cpsVETbIbcJhpwxm3xSFKkin6o7AOmNTgFDvCEsvUPaTxoFUoFcMTGFX8bu3BTIXQMFpLAGpKUkioZQWmX6kRJ55ySnJPfRAFFWKyKPCnFDqm85wXjkXhFBoL/qSgQ10wTfuwdMxebPEnUB2GQgRH5g/tT2uVB/JEIKPoyCI85VkBSA85rwJ+IxKtx5QzOEKtymfoTF8SxwaxUV85N2j4kMtmNOB0kcxnOs4ZT35TwbHQH56LLkkR1WmGL++eS7m5wvVZlwlHOsNBJ5G1k3vpcuuHiwc0yJ8hbmwQ+4E9zfCT+4QNuUnCUrRjsAjOhtqIVnwSiZRGKZSPG8VCZQ/PGMbfuf/N37lVf+feZ/N3JoNytsFUhGnnXN2dK0wF94lgr4SgiRxJiw8Id3PwDRV4yzlRE3UYxiBkV6NcpSTAwdMenNE3RMI4UTjbu+IZ3m8OS0ZEzEJgvdiSGYBRKuotgFEgoBRQRFEEPWfozJeBh4cKGWxQg4qKKYyqo6zGOqJOwAfihIqtTzDuJBadaPKQKxYyvSCEiAcAPkUm43gMwkRhE+1AEugmJMlJmlqGDxMgT+lzEjZDiJx2wAurOtPziXlgx2z4OQ6DolgunwzZOcwRrhDiR6EvgrWGonMZ1iuFM0FanQnL5fibLmaaFvPHvFNtoGEL6YmMlcmoNBJonOo5GEkRQceoviR0Cpbhiun/VN81AFeGsRHJonWXzzok5zJv4MERPm4RRyGug18uyr/IT0FCAwX+Aw0E0UVYCqklp5WMPC8mrmpHRsiAgKSDuNDHMn/kvqC0nosDaO8bwuaR8fNLDShI+AIFBdhwC8F4gyqowy+38+ghoW7IpP5cwQZDAmwquit1Qi0Nkq1wexKMMpt1g2hI4ureL7ZGDippvMETpzCSfpAbcOnV+MhaE0j7Z+HqqXEnCk+rOyC7ZlkCpBQt0CaqGgmS6mqgSe3qjoXwihgyBaC46DhY0MdO4Ei2oz/TU9Q05hFItSpDsMLyQeUhzwQ5pUFVlHQpZFEhf2hrNhdAVSU2z7bd32mQCkEg9tgLeOHQZqkGfuAYuSCo0E75lGXiApScMsbFARid6U7+fsFOAA6kQMpPY3SJOpj5AFNI2cJoKgIzKpbj+njwBNIAlRRGBwbMArVDZndLHROAArNhDRVALA14ubS7m3oBT1zhQFIh4TKUyznnbq69WL0ttyq3ameMD8aZdKPNfK0esfu/XNwNjf0wUJtOMX0AqTvsrnf4TZV51bdCUUIzE8fQvAe/UnGyDsFcod2kFPkVwuZyxS6m1C9p4l6nTg7lYmofxK79LEBVfHsM4FWaTMhah1iFPzRWseSmn9iZlFa++ZB9OtaDM4rl41wK2LHIZVp2kO4RXJAIkwjVbsR9QepxbVJRnEz3Gln8e+gUo0MZAGKXXRqbyCOQJfTxtUHTGuwdmQbCkHzu1QQrI6AriN0roTnmnpPKmklZQHMoiTnkhmPtgYbvoDIRg0lSYNKoKpZ7ykYloQrStVzOYrQ5bEOulsSzc+CdPBbtawqEXAubQyjTGGDQr0A8sVD8W31Hea4dINYknzANjy7Jaa3+PJMaSkCDIN2H5QmJFYFGa5NkK8TSAPx6DYAa+Aor4rzxAXHVXEc6OUohrR3MVSwHZSfz0SaYFNh23I3umKE0WpxjnnbUWC1x9shlv4IIbtHOIRtcop4vwzVW6PTyjc731dP5HhZzTwBrCJMGfxG+sSRVjcXSvhSanuUcRMfkuUeqnpxEKedKqSDC8E6vFt8kCy+ERAV1PsoOFDwhX8NKudPuPnSOmaARAsWEPb2LHNDl246UT3LzLGFGQTxKK6zO2JYwc5s0F5o0kBuOcIdwJSFQ3dZgw6clyHgRgLnhjHeelmcRBZpOdAH7sIPxC/0yD+Hmt5XA7oHI8OUPWq1WoSYJuJw8KQ8R4s6clB+yicB8eUZU+wLiIrEu2wia0yTyCpnSsmCeA7sx3fNOsY0BE34RrvIbhFlT5TgEyJMtT4TmDbMEhH956BTSxPhWDsRmRW1MjnImG67odIhH3yojf3OkJvDJIw+C5B44HCnPnKzM082WmXoS8O1pWISNQh8eYvw5wjoTIEsSZfNhkvQ84LTAaV5hfod7oOm1FmnOyTfsXql6HsefkhXfMfxc0CqCOHJ8wGVqCNAR9fVnkAhXU6UowK5JuoBZnyvWHMh/QPUW8EhmAur72Tei9Cl0lhAVgGZ7n2bwaXqIj3SOVgwKkUQKFT9iw3IiaX6awrvEBiBYuwkNk7CQDkgCiuIgihkpIQ9PVGY5RuMFbppDGTjsDfTCzBPySIBwYNYZbQxW5MCOsfvg4l7QuI+DqA4uWRqsTUQ2OSDgB48jzOOyEIDEyCYiZRIzA4iMIUv5uYMEPsOzW25y17ADHbVKcRQq9TC2AgrSMGnh3NsRFoskdgorqWVJ0XRWVB9FC4tzGpAMJ8RhBTAmyDcB4CiRDSbm4wHXXdSkqCO0hMl2eWoc7ewpjZU9ezgerf1vHq0P3KNVHof2r+TRqohGIHU34EOPP22nkYiqUciR8AGyVTC1CnJ3hONso20HWF2GhKE402d4h1iMKZIwFIAcbaBBa+krcNMLM8Y651JtIX2sEFskWTqI4AWVd2i78myQ2vSXwQYKVAAkh0UTKWh6WFAX6lep4zue2Alr6E63GPBpVc3nCgKdMEafClnR+ZgrEDLN2WalZbpCnqAYfSQL2pZQOwbWWA5d+CYITdRk1BYsrHxYMC9cSUSuluD9ADBOp7JXW6uY2NZofNYh4p4ggheb8x7WJWJFAfpk5LE+Rx7TojBM5DR+PIDYDTyG6cmQh6ApoGCDc/F6jl3X1DuWwAo8kU4+mnAzKPMnAEjLenBAGjx1B0zhWSgvcgPtCTAX0JmmgPuh4MMxyXBbwX8V2UAkWJ42y4UrmFfxmLaQ7Egsb/OBeJb27o/0xgcJ8OW9fbkENxmdvo18D1U9evAwqG96/I5sY+JhObr3GTA7JeatOCAWRxS5H4BOwQAEejjIw6onyIXl/NX2MVLXQ7GiHY9CnLZo8TCCaJDdQKIrSWIjoRzN+JiHAK4SuBqLN2EPdybM8AOJik90ARWTKMlhM0EgP2yqS1xzqIijVQSuIkbEEDwPzxksswlSagLmU9BETcU2lTyxq0yTt0wqCwRDTnrAKBUFcYfpLYijZ62hK7F5RXKrwyGAPoCAAuKa4DbFxLIi0rszpmDHdoHiAsJfaI6BHFsaJauwkziHLFoihMFlRKa+V2ZYbE6q/GJgVoUwLo36toQPOwWPUAaoA/QukVO+HbiUxb5lD8vsQ4qvmCDuJGr/E9YJGfiYauNsSniuyL4om99BYBR5FGGbrkhbHauwAm40D6c7eCCA28Y9epn+NYK2mQA3JAV97UHbBI0hhNqEmSGkiNtDzMuqPDNXhbADhJGHEf8NoIS8W6oqdas09psPBGeB22xygmQYythlDr5DiEWHEDldHBneBc0Vjut7pGMk7RHAwyG9DWnX3OBJZ0llyzrs6EBp1rEoCpznmjTThwlG1UjuAS0gfmmGW09dEbh622ERQS+4gd5tZSdBCIUq5hrCaKQrOh96EPpkK7hKos7QdoHGtgGhBuSWM/BIQMa/NciVb2pXsEBYxpcd0BDkxAiZcYWqkkVmcktB4ewArQha1vKaWHA/3YEJGVMrVGdZidwaZlQF4SHhhtybrqtnUcFYWhBcGbLZT7R+nUxLbI60jueKSHUiFhbJImu4xjrHT+Oz7pwMNfAXYiYScF3DtdQvSxCO7gFSF3tJCwZWxUwfqAcWOvDAQQG4JGFUJSlGt5zc2jre2GsRbwzjrDto01lXhJC0Fdtu8yG465IqmnM0O+rBL9Vf15MTp2JEaVC2IBHu6INGsTWsnTcBqKEutaRAMuB4SmGF0pS5DkHTQB5S50yigncIMojVUGBhVw08QouBgHaaPgFEGKjio9gkmnD+IIOzGJuyne26Y1J9z9SeotGFwqhBwAPhcqGT9IHvy/NBIgIL4sz4xu9jZjPjzyVgnOld472TG5MmscQ/EeKCQz20hWgeFTVOA5Fa4qAji/cE8SqBlQ8IJCo6i7DyjNeRGjAA1DRqLyC7VZADGnNLXDufHOCWbZrqQ2dtAc0al0HAgGGi9um54IYkIR7NDArthBi9C3jBTpZeuGzHwEOC5EYKCJGyJDHgfOle5owAPnaMPSiI2ekYe0m/TAqPE+FGiEAx3Oc+tWc7vO9TRbsqIkqwI2A8S/PfElE0lYNA2wyoI1FqJKh3837V6m627hx9j6pjNBdHjyFfLIV3XTdGxYqQfeJp8Gwo3Z587+oKelfm1FuLJBma16rbbbpfKdu1i8MxPWifc8Gj7mEju64u6L2t+4CoYdZizPPUeOy1SXEFUJhOc5MxVh8EAg2waIEnqCSaU5u5/mzKnzAdtgRNC5KuQ4hEkwMRRELCDF0enhVQ67wQZ3KQSbP8Koh0vgRIDDaTCYQAlwsQWclXOvPbe3pPQGV4Wt2JkEbt9a6p1EFN7ixP/7SkN8CUTndtvORsKWstcXPXJwIIpeGoOxGmMsZWlIB0DMLXUYCZMnKmNJaHgDJ3AlMa6VmsPVnOZrx0mMzI3I2cvKs4vBLH6q1qqyH3AXkiN3kaPpip2L+uXX2aI74rWXdreTSlBssP1VIRxFeJW5HLvCIlj5+7gDuxlQPyQ55q0OpYLUSEBOOiTo/Enuv31R7k6YXmNJccI8RKaOFuyg95PlsNhfnYUKYc8aFMoCGNMtdgRhJKpM2EBNAFAnCteRDCbq9LegkwB4Tr9PLbCwr0fQvieZ/JqzUnIp817Kc40tKDpv5q8Fbg6yDgw6oD3ASOWKkCiVH2COtuJKYgMuFQWg5zSSKwIjydD1TIU2eDJbcuP1PoN4i4Rtt/6nMpFVlRn9iaV3ZIbFbSDS7pXYFfPfXLYnucwLwLuJ9pwD+sA7omxggfjFR7gOCfVVjIF7D01UkzAJ9L0jV4QNNWEmthmAD+HABGKuSQmMBMbAkJJ+8pTe4KGgMKm09ddaQG8mZo4CvLiHY/Etefb1DLt4A6h84H3fOdygVWnEkI4ngCt3NdWsbhixd0PHOxBzIZgEYRGAAogApnV20u6RM2JCseZTwSbs0xCJW/IFmaRUF5EbsPETxc/d01+ZppPy8b2RJCnCofbYqe55H0W8coJKOaA2eSYkiGcOAXUVrJmUx5EijonzhUBgmwwZEEeN0KgisFCqIzkUbeHsSWbAsKXBeEEZGk317/i3j0/9XTb6v8+VslUn61/vwt8CedcC6RpiL2PLZYDao0Cb5h/XCdkdsKj8sA9VcaHRTtBRsjLYhkp3G6UzD5xRpzbd27EfEbZAwTW9fpstqwCnxboQw1UBhVFrJ1uPkb1oGcmh1k//GBc+0JLqQhHcJCzZi+C1JrAqzaLf4GTNSoNIvdUi4oUp9L4mnIsehsTJh1IX1zeaxGk1qDHpIuKRycoQtA4RgKyTuyGAa+1Fg/SingJJEEN3oIkoYs3ZlCDaynDQUN6b7i9OCi1EaqmJtCKukJn8oZdghKOm5P+haaUYnZo7DjIBjlEl04mH+sDLMScK4R8CEOkAKy1guXS63bM8rPjPY26uSLwLLEmYzSZ9tyfpZyB2nAGDn+MGiXaOjL6q7tYXsEHED4RaNAUcl63NLQk5cEN1WLCsmTbqGGrqTfpyjFUTgnS1kM1K0wlMpnQTG+wN7sbMs++1pplmyXDB6+N7vQE0yo+j2DZNL8DKH79KHxKlFARDGvshW4NTDBkiepBUi71Dp3EvHxFCmnW8gIrguGSqJXkry0/pFt++45wd3QRVvgiDBp6AHSDWxOgSvpUB8goq1k4ABVd2ZEMutAWKanNo4zcgpG90OkbO+qycQS1AXCOS4eD8o4yIRHHxndQExrOhm2fDpwIg7xiwFaaxYXsWc4DrroGcGBDIQLMk0DPpAuNnWwuMbY6Y7H9oT8otaCzohhqhPew5M/u/ZInYJSLFOOjORtwQmfPGNKz4M7yqdgpuA06q44gjIiiBGHReQaKBWMvysmT6LmJoc84HP6CK7mFvIZvO9bPnnDkkYAKTg1fKm0UNK7ING4I4LWFhJGAmEDLskSKV2/fSDMjckn3TL794QMkVvRF8Y0DAX6CPJ7Y88wd9figDUX2Eowwh30LVD6uLjXhjdZoa/ciZy65l0FMqGGICiaKkkRo/+EDbKFTCXVwPFKA+9BueqqoyxSaCKdWps/eHbsHDWfDzkbuGb48nIO7auVSwFgRvaCKkFiKwl5SFkUKSlxvLaTFtARuLaosg3Q2FoLYoNkKYBWIPOKcoh0inoWpTIWZjwAevwmyqUZkrREcx5FGWoojFBT7TYmLHQvRdqTOv3Sho8BjnO6g0YWgwakFUbYiF5UWkTcQ2oHcgR0jtrJh6Cw5yKzJvf2Flo++Kl2BuQkpOmRx4ylxNbAosp2hS7tdIgFnFdbG1yADrumI4eLY3/CxUbs+FllmEhRSxKlArivE73RtT8Gkvkpo1wxOHR3g2aDIp9BxpbHYTR5aKFeJ4kQzeAMm+qTGPlRkYfWsiVuL9tWJ0rRKcwk5bBuyVdKefBMmTSSdtLmLyVPhJRyKYKIAiuawrlx5k7tQRLEjigxyDM7kQWR7w1InbbEleeFkcQJqp0/d4mVIX8h2tPIVuAllHPi7CsEJZ6lDVqQUCkOk5pMJYsMaW1PlHq5r0QS6qNSQepnbg5xa4pDuABi8weE9l9urJUAArv+8upw11bStFylw58JOxyjLh4hRJAu5N7YB7t7QzIxbKX5raX4e+5Z6fRVU0R6ZQcYF7DdpU4tzQPjl2PnUFqLZ2P33Bs7ZM2+1SKrxWgp+pcZ4286O52VpU70bzkxi5fXOst7u0dHh8mv48Ofdo8P41/l4hf/lTh1LK+lz0WmaHItWgPj/8d7Sfz/8rNRwdWlv+++GZ69PtrfG0Zf/MP84nant1F+dbvT3yi/vN3ZXM8+Y0iw3en21gsx4ufjv2I5tjsbeflEnqjk5nouVFRwfb0qWXQ/urT059KLg7fHZyfD/Rcvj5vyrT/pmQKuP3lqSrj+ZIsSsffEFLH/ZKMQcf1JvypjWjSTsZ98rCLj+pPNWMa9w6Ph2e7e8f7hq7jb/uj3+9t4QFq8KQ92Xx2X/jnoyEfCcsvfaOyT9XWEKoJBdQaLTu2Ifn8gaAirgFRbwKi0CN8mUTZ9Gpwt7G4W33wTH4DQ/rz9/oagcRShd4F3r95DVB7JbfTnn0u7z3dfH+//c3h2fLS/+zyaB8fz28nS8VEkzNujqCWOd4/iKfy0u15e/PHg7ZuXycWtpUjQn/Zf7R6cnRwe/aMsvj5Y2ot6f3j249Hhq+Ozo7ev8lcXF85eHh7t//swvtE1ru4fD39KltHC163aDbl2mGrZq0tLZwdRu0VfHr6Onvquu3SWfnjv8NWP+8+Hr5IVcj1aoi8m7ztnHyY3K7N3/1nr/DK5W+tEl8a3lzc7r2ZXk9XtZBGZvu9MF9Orxc346nySFr2Ynt9kd+N/88nN7fyqE916Er/NfNHqklEgujm+uZk3P7eaC3N9Ob6bxAUW2euzJ6dXNyuZpIto8UqLRZvA+mpnNo/+mz8fD8JF83Hj0aRE9OTpKHn0dGQ+ajyZlIvayXhlcj39dHzDkNX8VlL+NL05ihsve+L7zuXkaiW5u9qZXC4mnT/+LKo9n/46vpmwgmdlItH/+DMRvXx88fPkwvJs5dXRW+Ki5Csmk4uF9B1xWeol06tfJ1c3s/l0In2V8QTVJ1Hp88q70lkct37ZMOm1ijz1z6ZF0kY8n5CyX88W05vp7GqRdFEh+iIandHnTi+ni2wExrezcZQMwdNoa13PBE9/joxnn0w+3UyuLlaKuZK8KP/YanH5ffR4fjUa7Z3Kt5Jl1Gyf9LnKsEs/V/TEzWR+djO9nFSqE39kPvut8f6kYOX92xXB4vvJQ/HD9QLNJSIunq0Rcemfx4tk1qeXl3+ZXl0s156P/91NJ5cXyaeKLrl9dzk9P1tMP1yNozpOzKqcz26vbhaFJpUoO3uHJ7EuEyuTL4fD19nvF4eHb4bJ77JkoXdVtK6KzmWUrmhdps5lFoq2ouO3R8P8pYeH+fdPop2zLPvnUr1RLX0V/xtfTT+OL/PRnrVfejHqrWSJLsqez2fXtZLxpUa5uPlr5ZIeqZeLujT7fCRj2trVPkuvnaaFRp3H8Z6V35tcRo8nEvEPx0WoRxMh+UfjItmj6bqcz9+ocje315eTcsotZvObaIFMrxZTrzHl6kuAOcn+ezu+mI/TEUe/3ZxPt1eXs/NfJhdnxWNg8hrLQ7RH8FO/HHPF28tLRRXKS6m4WatFY+3jKKl5/CuudSZ+en81e3uxnxZz7+xims3uy8n7aCWdTz/8nKsA8ZWzRNC19HcsWP67lCj9u5i48V9pG8Svyp9P/8hekP5hvCG9ULwi+TPt/ly86Or43WKlFKnznfmB1WrhaPj0O3/LnrhaKYVfjR6LrxjirDafXdx+LHsufsk4euxddVEfr3Xexe38+/R6xWiAal3yIZHrW9WW6/zPTr0pyilhytNbMgdVfifvy9vri2jnPTu/jOZ5tLPO3kezP91CFzeT66wvP1zO3kVzvqExFqJdzW46NT22rggusQpUUr1CEfq+06Mft+tZs+vrqAZX8bbcjdo7vV3IF9cl2asGkX0ySCbNSnLth53OYCta0y7SIv8b3e3sRHqwsd7OfrsqR3zyaWoHKpS7sqNzkURP54WN543RS025yvpXEXKteqshRnm/srIX3/s+asPq8krZCh+nVytba8Stx53uanXlNt88ELx5/GllnXrzd9U3LyZe7+oXC1o6+sfnN9NfJytVBbVuncVDpfmyH5Jplr4ses0s3brTeZT8bdgP+eg3rAdjl6rvN0mBRA/I92ebelYM9uSDsUjxjCpe17TM4q09fejTWucuen1R9jR5xSh/4V00Idfjj9zlb00+bnlj9tCn/KFPlYdO70bMc9nfecnTT6Nqq+aWwV2zaQ2jIW6putVRbZzU7DLKNGUybmbN0TQjDCNnkdhbZ+OL/4zPo+ekXS/rWThItF3/4ziaPOlTi+nvk47Zq/E3u+vJvZ/Hl++je0mR//u/aIgnW9p5ZCxVleyVpGA0O9c6+S9DI0nu2u+Uz9HPrJo6cj5AKlpb3kGr8RKfilcMm4vxdTyrz27m0/HFSjzFZ1drnfoGl21hNZAnaohkV4hvxZtHv99PUL8E3Vqld6hUvdhJLbr0cwn8YTPX4svRYL2YfOrsFA0cd+KOeetxrtBGghqXs45OlRhansSg2LGtSukIiC33ZJMpzP0lqVEgNUZ+i5SM+SR+3bvZ7HLFLJ3dOruZXYzvooeSsbm6Wt27oz05xp3N7So1lRbJR5JlOYaDkiqkDZ6dFhRoUOeHznp1v0ja7bRsz1Fs0y8nuGx53DCqbmX5V+MxUXw5t4lSYSQfeb7/YjlbsZPXlpUcPIBKpj2/08mKJR+OK5x1leQDCTRrqeJGpYoBP2ZXDajnzmfXd08uJpPr+MdKA3owzz1OI6FHp9kcHpnvqTyWDdp02hfFs6lcrEmTT9HaML36EM23y8tovs0vJvNFHVv6o8QEkgLxApSWrDReVWwTkY3LriXL0GqlTNy88aKRFFiNl5B+435y73R9lHRLfDawXGqM1XZN5T1NH+iORsW5XrNQMmDzgvGIbZR7nGtr8SBPS/ZG6bherbWzFeN6f5toyYtxjJwYK3zZsLGNux2pJqa9mx9Llo2evyczHCKL+sMked9a0nrHR7t7xehYrSFjRYdVSp0arxyls7cAKYtNoYGgNVtT0smSjhZ0drNYd5S0RtrklS7ZtnS7MTYe76AOprt0Pousl/ntFdq+a3p8/OLmKUpkgqzTW2U6vai9u9lPFas1m8FxE3ct746Vw2SLzUH5FK+8nc9Tm9WyKKSaXwJPJ2pfiXzn5vDVVfLiWBNNVr5ky45Nswp4VB2upZqVDO/HRCtFCseagUixMyIpHH95dZuyXc3nvku+2NpkiQfBg5wwtWM7NHPOowE/vbqdVOsXr1U7xWsr9/57O46euLnr7EiX0Bicj1+Y6bbZSGqKkt3IMMJos817dq34aFWU5t7beE3cKNFakD+fjtzzn2fZMM8Mnnx9XuusNL+5agj9JC61WDEh8l/H08vxu0TzjdujIs56FR5JFKpoViaDLP1ecX5olvsun62VghSUYnZGNBELYQzpzS28KF5dmSrjoIRy4xUgnlSXs/HNSroeGAKVdJe0pesViR6ILNpEtGqjJG+qXPlb2T/VywZnxfhy90mtvR53VshlJWrIvDtXa2+uS0+0bTZKnoyvr+MDtJW8QmvZUClauLQfske2G++IQe6V+eTXaKGd7MRn7uVnzurvi9XF9LFo4pcIX7JK58KcpotB/VED0csVw2wpG+XTeXG63V0voI6b+WS8uJ3fcdtdjXiQQS3JGvt97aZts5t+mF4lppt4u6M3MXp7yrbzj+PpVbSrVdCCVAEz1inL9KtC5nVVrQIKvLudpieyI3NfTfdSY3esadNpC1T06eYOcg86Na1iZU1n17yTSueDL/00WD1sW0htxap9JxMknZPVxdO+2VgwZtViV6sinl81WTvf7TQXscrYKLa1ap0rZaoj83Hzjas58rNIHjhbXM5ucqCj8qZMl7rNjwIatkoJnY8vLgq0L4d2qVlA7Hln4/fx2XGpW9ZapXKYa3lI0ju5SR+N9irjadummli+VsUeKs24zWk5DSWjtrNzrfJd1g2Wzbw6bpsqjSHJ+tJXvpkb21c+KMXbb/kEsdkWg7q528Y4bv5oZX1Ojl/TZaFhb8X/3kV75y/6BWTJsjtnL6huz5P5x3jfOLucnUf/TZ+yn0fclegreXohQWnzXZ89ZsiQSAPJipflX8eXt5MSG6xsqem9aXnWcfckuRTp0ku1HSczzE+XY5eX5fKsqaDgZWBuwtc5u72aJuwGGpUs3pV51xmvK2W348km+rpkbp8NZPn9ZB79mv4ezftiHTAAZlsVm84TuYRFkZhiutwcEb/N5r9wWltZPYoO+r3RPYYy1yzZKPVDzDF9RleoPHdPDs5NPa9kq2UVKlCNrHjcqunPXHFJym03FNrsTfGU4ebH+qrbIUmy88Vw61p6+pE+Eo/cydVt9OGYO1g/BMlPSrKycVWMvy31KaHiAiXm6lMczqwKcN/xh2LCRlvB++kHk06bMywKju6ajW2RwuhVPc1QwwwKZFzSmITG9taEhJJDuBrfJAfsDdNix5CwbDWaSZzcMTnHCfSTXgasE3M9bgD1VZwoAYjS5seHfWkxDk9MSzAWWFYAzPjKcpHeXvp/a0xlWU1KAgA=', 'Soil V25': 'H4sIADMTcWoC/+1d+28bR5L+XX/FrA4HixvGJ1IPPxAtoNi0rVtF8knyCbsCIdAS7fAik1qSysZY7P9+0/Psnqnur6q7KcuJg8OtPJxHP6qrq77+qmp9ff3V5LfxdXJ79/5mcpWczUfT5E3yZjaafkxuZ+mlz8mH+exTMr6dLGbX4+Tps90n27ubvW6yGI+Wyebj9fX1tcmn29l8mVzNbj+vrZ2d7L8YXO6/ODs4PjpN9pKLfz36MJp/Gs8fPU8uHr3dPz19NOwmj34eTa8X6pL6x6fR/JfxUv3r4tGbg5OBukP/68d3f7vcPzr4af/wUXr3i+Pz9H961S+ng8FLdf2nweHxUfrHbuuX8zeD/bPql9PB4aF+tUde3RoO/91NjMb/+O7g8OVl2oWzd3nL6l5cPDofnJ7lrT06Pjl782jY7Jhq0NuT45fvXpy12yTta/n0TruVbw9e/PXdW/NhvaFF87SWqj/z5jPb3He0tN2ew1QgyjsabWmNaNw2nb4ZDN5aWlWNkv4CvW2nx+/KhhDNbFxiNnJLm+xWk16puW00ojmZ/eYYDfZLsaNHixZ2S/N6lvXhGL16iM2Gv9gnJrS+mP/FXiKtBlQvtUh2KnRHZ7VSaDYkV0PMz/eYn28Kejol+2eDE+bHRRPVtyo/1wLM58o6UbUG00Q/QmN7PFHXP+qePYuCFY5gq1HlSNDrzxx6emaLLqCG9YUNKyfGIuv1vNV/iQS8z1725bzk9zSaVE4qPX+OOcXj1R6ScvwtEqTNj/6nZEPZ4k0ENfrNv8wvCoe1rVa4w9t8b0DHqeGu59sy2vmfPrNLjDOlp9DmZ+ngtlSytXVFDHPdCjTgDBuTHGlNnurhbTfZOuT8cSA3NnJjgMuKMc9Q8FsdpK5RG4dM4re580DKeXtQ0HoHtie71abnIDArobNCb1nwW9iq1kzA48PDwYuzy1eDk7ODw4O/m9Ikd6S2fJpDbqPI1AjrNhyAWN/UDW/3p2wWuU9HLV9qqmp+R5Gp//Lk+C1Tckqp1hpH6iVyyWiatwAxCMUc2aVoGnTYuAi1wslusrea1icq3Qh75WmlEd2iv0n1S1/9rO64DUB2b/xnQtzict3ZPC2qyRY3hj8DbZTP0h2tJeRHWW/W7K+4L0ZNZu72zCZrX8t/flAt9pIZ/zfXA+Q9FtQYS18rNNQs+8vqzDSRZWiHjAz3//j40OIj3YdxJ8JKnoXZokzjzwOKvh8LkQ8wrMRWjGofc+XFB15pfb3HwXD4mIbYlmPBm7GMBxcUFq1LchQsoqHq9MWj2aqaB0MNY7DRhMzW+vttI5C9o1EGrDZmZB9bp00e8kh/g1psrE9QVm39Xs0jpIaP+oJz7w4zfElbj3vUEs+ajP4Feql5f6KeNJcVj4+kjPljTpHm6Xp8+yF/oin1/A94mr6mCWqZlAdiED8o+LL3UG1Z6mxw+5uFGx8YvR8Ll9wco1u4FGAV2a51oG0r6MeXMGvRsdvvwap1QZuMQ2RjY3k6lNq7pAxFtncdpwVhMC7b7KMN3nuyaqPid+zufbNqv1m1qwJ0exaT6A9t1Ypw254Vny5fU2LT2/diC8uH/j4h3hXQEXyNYenZvb9ZjKbrXjDgN/sn/8v4R2QElVagLAeuebSzZQ0c2OKpJnuna/YfnxTGMs+ctC8aMa/bol2sWusaqNOzk/3zHwcnJ38jWdVCjiLtUVWN1z5mJ+5SN7t64E3XpZcV5QR4CDugMRpv1OVncOog+rFECjoc1he3uyrttJse7G4DdY0aJp+Bp5YPJEsjXjvLR0HnF6T76X0EgE4yKLsx7seidA2wJOJ2ze0YUeQ60tQXuC/9YQz/z3Exvivo0eUAWip8Zc8Orj84tgknQvCLsoZ5jQ8KJbwHh2Rr1Q4JZax8GXeEiMb0P3QJGrVaa9ia7Q4ijTV+mm4mgn9puHcFXiXVDD0+FHNlPcMnekiT2b4tPZLwYMEEdBQ6ZM6I08i91CSZjO2N3M9aUF2IrAcSQPoHpAvi1TlWHCXlKBhRZI143VjsaWqga9+5ntY4ZF/2yRMytIIPocBBG2iJuOOkI+Q+gCX7fc9EJdSce2Rnf+FPRXUjqGvNv1ZMxtl9UP7Ck1jHFrbgfDZeLfQKvgi8LfXFAs1v6AeQm38sC5IEcmXeAa8pIKWCYWXRqUwkwZuhexU5LOdEA8WhPsjUI+FLhrMjPU1jJqBwqyHInRHiucCAcZ17xCUKgbMG+pSI4dJ7W+f8QXCnaBBC6CR13E5lChhwEP2PSHRN2NS6QefBVs79WbxmScfPgaZSw8RA7chmuGlBpEcls+7l6XEQx6vZUF5aobaJJUwXx4vUDBmqnofokPCRi9ETeDLFi92NPAqkXgctsVv2Hv4/uwF2olMMXh/ZDIqx5tMMAXWqN1wNUWtLzkH6Ao5P2cbD/aOXUd0ghAX7xydwoosNZsW2I4ElKRGbqzklEhzEyEZnJ1r0hiNdInARV+OIcfhh8iOjaJ6aK2bDNobesyx21/DI+N/BTqgnzqhDelQWer7jVn+LDrh5bs4ckLoQ14/Jk7EOjJUJFrqT8zNEUqg372JgFDo1F7WtARrDYsVx/Sc3PM88U/MZM2He3WdcYwmcSoJzHdJJ9Bx6x5qmG0nrlwEz8sN/vP25e8hxq2Wayksoll+2xczHiNhcp/AMkz1xRBbL5RIlTvIGFgNbNeBmBQ102BAzEJxErWKO+e2k5Y74C6cnC81J7Q6x82iyIC/pQ267VBooffWUvLrzcPz+UAQgvO6DyJ/ux8QJLJz3CMkDxGMt8fHrjdb3kFXk+FeRaQeHf60k+j7RAEuQgu3U0iuzrZ31CuCdMIiB8NGR8cSjX7KzzsYN9KNPaS1tJM+S6otu4msTGAJS2xPzhLymUUIE9cSRqHujyLLXIJAtJGjCgIjrjEMSUVfY5HW332PyIOxAVIuv6JZa0uXkmRyc+RTa+U1KrSM8TBNx4I/aBF8rmgCDE4VelD3OkPReKankUOtJoMuzGwA0IpcG1UaUAJCyheXxoT0ploS3bMAdJ1ZWW7D6bC+TMSNshju1UDi2iEYjohSkfeUzGk/NBrk/sTFGK6qrXWyxsyK2nmHaMnoAlRUBETD6ghYHtbqdNrd9LdO6x4AxWivFwRPg9kBwDIDGGLEaWvH0gcw9y/cYQs7Hi/3p8Ahbs2j0gRvIIITAjguHSDFqPiBeUdea2tBn+kljiqEfVkXGp7+HQCqQvpWJZ/KO2Cz4UXQGfu9JZLiHksoYJ9yrgGxsfoOttl4YQGMen90fMqVTCiEjx3adwDHJ9xJnRDFoLL34MB0O0rBGGvikRlo5M4YxzbCBuO8atLVv9Zrvoy9k6AEAL0Q8rEh1p0jX2xRrm3fy8uC1HYcMoD9xISpfJK7HQR3J+bNukoxeOSDVoHQq2gTS4S6IfHPentCq7QLnxp/2Ra4KnMyL4shwzHTP/J5bcmoGmeoLAS105FC1rhpTk03ZqqYDekWUFdpooAimZifx8j/d5VOrUKAzQqNAlHtZLii8ALQ+9u4wQMZ8obgk1vGXzpuWGOcINoAGAtVBWqnwa4FIyquz1RszRzWr/WiDCittgmihVEdqdeUExnFOiGg8NDeRk9ZunrWVAM2G7zHXqwjuL4hxh9QWvRp8pAmAVrTYIqgKYBzepJtn4oMJNLqQyQyitaJkDGQGgjGQQZ+61TBnXYMoLMNh3YUF+AMdpToGawh2YhCnYkBUfQaUxs6b/lDRq17/a4SvzOsAxPCXgM1V8spYNDmf7BhCHlokjEcqv6SZh7trIRFSnCDghkdKPGeDseq/MH3sHLLs7hHgGjBJROwwOBEHy3eqekxxpCYIx9GJJo481xQjxG7DSuDZoSg7IavVu4wMwArd2Tdog5vn1K4gnbQs67ptUVNLrQEvxEidQvaAyrVulRbokEhVLR8JJA1ucrHZcr3Tz7Wc2TCvH7kIjPUmQnBBEKx/1CQHEYZOKf2nm5sWxkMAgAXk/ZA8aB9vrqyIsSuv5EBoAyotOuB8wBkBQGvdAZdf0HPk3bbVf9+SJijVtAAqx+cmcCDoRARj9iUpx/lZBUx2MOG2m2eJRPYUjLBEmPktYf4hqCtQ+6moMWMoXLAkNaWW/Cf8ZAWeeYwQYEQz1GXM4S07KPFs6MObYlg3IHOrOzM5CG0WRI1yFDDItkaeP1qy27Bl1SvVL61A+/H4YuUre09XHsMYA4rbEkBx1UM7DxiG+6rxNvKV+eroxxuKvsecb28/NBIZNJu5fyIgd2e4cpIZnWCxlcfEJ49t5EKyYTQuFpS37wwy4yXaigjWAYjOAv0IKoysmIi2GRYOKQHzYE5saZQXL8FKYMAnpDXg0C93fGhLWPibs5iYQlpa9UUq5Rg4v6TgZje10AMxAfgqyH8Msz6BsApR9BqKNgHxdyRVwk1dAfmxrQwqUVrPgHREGPFEpWcodSmGxgEiQO01gFJkEzvWAZJNa/jDiozoHzL+ClnS7ohpULk1NixNAXMCOjFkriGjT8/nLWEMgmWPqiFxuN1+3ClLlHS0gHTSIYewsAN/gIibQDtLmCRA4SGSHn+uAbgeIeoVRDwK5Y4jBX7hoBKGokeWMl4NN87kMqnogkLVUToDa6yJohAcyVjDuJag8ix0C0gE03HAE7J0ojDtepGpdi2jietBbN9fMKiIyQEwETZrNnrvfFobiPQ9KBadtB+UjLLMY4ToeEF2xuLejZb4S1Qllp+6PEIOwmdhCb/cSFx9jcec5DLpVl7vC32RncQL6zRnDiLeKS/VM9LjwQAcNXfBxEFWXrotadpNxOXDa46RbI6FRYSVOCNOB8quCdMAkZNCrUs3fmEhwklMbcI6pdvPh7hQFByg6zhCHNHxzfaOf4wWaSZTU8JOCCZtfX+bedoOAuUcHAjO0YZjTcE0q632axKKHVQrgZMKxZIdi1FyatXnLX0nKfhG+Ucc34wT5q+B+4DwA5AYSzuiRYAZcJ8tNJuuec531OU5AbeiJONG9QXgkPBddTAh5WqAmduFOCMZ8CZFUHgMOUcoFQPnq57a5VXNRnhV8NRKBsQHy+zz+mmdb7LgRaAQUJKtHSB4lwInZ4APJvPLkdBTmQ8VO5UCO/W7Wyxp9JJK6UlMD6W4HPUEQmgFEqGDob0+k9bYV8DHbdOYWkAnx5HKG/hCN3Gzyfnih9/gw98xfLjtHYXd+1LIondCOV9g0VkeglEzy49oxkj9QToAfEaFsKg3ebjthhFlrhhi0bGpfqGZWmQSRo4tmA+pAhTF84KiAg0ntr8pRe1IEg1gmgDSEjmcUljVVrSIhc258SoS1yLtRGrNNSwS343YC5Gj29Puq+xQA9E5wdv8OyzJru6G3hCHE1A1EKYalhcOViN1/sXjlgSQwvuUItmWpi8SJCriY8h0KknSYwvLcwd9e0cqO7e8gX4HryGSZMINbGVUUgEgOso8H6oOULA4+bvMPOGiPXFrJ3Pw2ippKrVjQfbXyufrnA35kyR1ryRnMLmhxMbYFpedEqQ28GLSAeqWLB0jIyy7GTZJdBmgz4IIEDS3NWpZC360mXX0EGB11EptAZ+c/olqI8TJXPs0FNNzL1z8pzt/nvP5OIuYic1ClrkbdEUxzs4NWUb9a9dUWBWkuR2D8mgSjXxZj5veyNZmrDK3HAODhDfiQJtb98cMlAE9AUBne2dbhTwMXr92n7bR2N/WfUCiBD7iYF7KK+bGzOhnhXX9wqVwUkJRHPR5OLjUizrliA7rZmqSZgMJoJOQaRRhdtXTcwXU2v8hqUeygsjpfgBVk5wanHJai7pdoTTHjZykJJMd2kqornso6UGbbyKKJvDC3Fu+F2ZfG+nU1V0fgBgpHYCdkrwRSv/Asi0NAsUTV+ocbuwlidbTf5K3gmhKZli6fylAhM3APykAUZZT0y6t7WKoPFTYrUBx9jHGUQXKkR9QW4xBXOUn+URYJevgwiNoAuhmAVY9cAUbCE+YKd1NhYIjSKx+hq4Qxol3gNhCS84NjaxHkDNOJMnFDxhfJOOZzozmpsaJsF4YI+/KR0lx1mmyrztolwp+hVo7sJSIZd2DSSJ7x4/ddWkV0F7ylAdU3WAPNsgH4TwqCWv3uTNJJ4mk0eHCLAQdSD83Nw8B3moDx85ryJcbgfoiSdcGHrq9ogDp/ua3AOmYDMe4xUcMW86e9VOCgZJ5P1dBh2QViPPJihgR7GEMhif78QEBfR4xpoyYOg82Y9z8htbuguqnJEzi5lVwATwxy+NcEJ3vi4oEv9k/oBokmOOFNToC4EkfhA5d5swWQC8QrgropiiznjcAF8rQw6JCJiGUgKhsiqO/rIEqfSCLKNsDBRLG8d9Yy4TUTcjRgjAinmpx4UGfdGusvYOD0gTQosiTJAo/eDb0LkbjzhTALwJ0HremCPLAuI0VnRRKKbokkiNc9TCDiuXy+cC5ql0cL4QDUugLQ/tQDDLNewVcJxxNWaJyskxq0pMATGqV5iW2UK7I/JkOc6aNovNmDmZXRIAy6i8JqEoyiDzhKLpyGj3mEUDTMO8ieWszerSJ3/ksMnj2hI9x2GmMm0iZj5fACnjg0FRBIEILbvRpLAXrkmuQR81nE2uBZBCCvxkPRzOCzJlFHMpH7gtV6/+u44ZF8FkRafd0GKli6EPNRXiP8BkeDJiVUIRvfKX4GfKZI6Qf/OIImohaSx1Ow3xPTX5wSEgVQ9Zszg3iMbEH4n5hNVLcuLkJz71SToahbSDaVxI+a604HV62qy8GoigorRmb7FtoCTBVVoKxkfiorAeSWrD+UIjbwkLGPWwuOg1mzLEXNlLYNc9ER8eI5ybxhUDZVh62FTksmJ0XjVGfGnRFSK9C7hOKoraWeAE0WGLHXXXQPA1fIXDEnrEB6xRpxWsQUo5hCoJm7VWbOQrUXoF47NSM3uyn8BhkANGgRHKIMMk/t5HV+3CsBjA7kBrHrGzBUcTN0YOoyLZPJVljdXrliUBkA0CgkmYiEJSDEUDSEML0QSv1QhsENo/8EUlAOy+az80zt+Na3tV7KbPGcAGfPpww150VoGzegVb3jrLFys5HSnIcCE1mZN4jgma7bo/Fw28yVsmWFYtiepP3Hn/KrtXBWAB0OJSovALfpokRc0r9xa3Se18TSlrAdoH1nWcGUBI6YwCCkpQ9lkbDCuKmvVNshASY0uaqO0GfjMfgDUyFBJwCUiuKl+addgekEiSSIkLTGLFZkFnbWFQUw9K/R5aIHbqQ6IAHXNjbzk1w5C9pOJbXhjZL4HRk1DMT8HkBiqgmA1CagG7AjrCEHhY/AEOaDsxduIcQu5qaQ25Q1hlrRQ+iAKEd2utyFFd1Bp/wfFHSbSZDzvk8RlsgN2RrSfouDHrGeCX7YI2HhEN0UpMlGEDAXlgoEa58TmTZWnaGcpaqk8dNnWJi+eIc2coPNSkJAzYHGULMr2Ecnp20Nwxh0FXwFIgHFiSaQwlxGVlyEG2RiKLVkOa6K/b6SAIVUc+wuz9QfZGzAnZMZtY+FkRNjguMyY+GCRqWyVO75t/6SkJa+czcXv93ztT7xsl7QJw8Bq7BLBr8VcS0ksQ0J5j2tZDyLKcbMnaOhZi3CvodnzLJ5z+dBxYFDoxhtYQbWQgdJCwoCzn2oNUxiRZWGp10m6FLzIQQovxJePKoN2CarzqzPsm3s+QjBRkqrbi6w4OEpdq4JdG4GdtwKjuEHjGR6lUxPvFxCUqlxUBzo7joZgJucUkiSzOho0TfYBdJmMRQhEc846HweMJsd+AYQeQZs+pTPwsrFc4nSvKzgayszJlh9jAOAPglUu06B40/N6GcrMgJUg0kIgtJ1Dxste9QDsFFtzGS4tblJieCWW2Vj1r0uGoeJrwbyKsOcNGZGPixG2SF5DH3yEdiuKIxJiXfUVmXdbzkhuqM1bA7FJP6+HkKQSESnNLUnXrOWLu7K+bvNUYtKkpHIRgrBeqeRIXkCMePAVoFR5E86Jq7kWvF9ja/PK9vZVnmvGLGVgbPBRyKctS4Z5q5qHUiQBo5dyZmarrptPzyI0iEw4kKGuAYJFkWrS+Vag4UUaBmC8QrOzLthzk3rLosdJ0REeBme59E+CQz5E6BjjQGhWAiK8ktsXFJlDShTRBGYoMPrNOExCQumxIUGUVpENk8lYDDFmZdbZFXjjF9RpTmgAcxR8lg15cH+wpyCUIptkeFwX76Evfs4bp4xiRB6CA3hE8BCOu4gQKGpNEQmnHY4qaJQ64hsokhU8RpIZYXGH4MhfKPQDgQHFYRfBe9EcwvY3QKTmW5UcpWAIKbV82DzyUIm0e4L+qFZyLLSCQumleLOiqYDnZHii1013JqQuy2W2LMFJTCANgXyzdk134CwcNoiNHESHmcgb45wrd37Ai+JWXks4cTb7slwOsM7fnsWyju/YXi8tPk/pFCcQXpD4QmZoyuEw2msCrmA2RfMfkLsouY0rMTKUeBLHBXVqdhwC5Pen+R2QANA8inDbtYWRENPPTMMrfeBWBXUPN2C0d6oUA9av8lyUiIxwYdA+gw+5M+BDAEqH+L1ldIBVxmIXLQQ+qv2h/wPDpk9jvc+TQBqYoPyHQKRLCNLGZeANVxFiyNkvRd4SjCih9gpyBvtGldOyDX0qAe/hFEPCWVwxm62J3EzCfbm1s4WKVfZVB3vqCDxlrEObcmDLSKiMBsb+q8J1JyrCsYWlZvFuUVlGCefTFsY8dvgTiALBVuhDQwFABCsoBrDg70m1WDmVsMN4WEm+rHFxw8DMJyDgDzsp5XkH5TM3aUeYjrbjKL1Wn3lwSlULBt7sfOcxTyRTkUnsZClwUFSzgAoLQQR5TyrzS629t9IBG0fV+gr7cdk53nTofwxyXo8eIDviKsD9Pz5NgeOY7Eww8a+RMC34JjaXZ+FAfM92WYi8AKB+Vj5cS+uFzFxpZFmnoombXneYhXinS2GEGcyO4e0Z0k8ZFYCerdyR0BkMUIqWZvwgExWJiZCOorn8syR2YpZ+JWsUDx5tSU+NZ/YaffiNxFDJUhPB8eTFhWE+TrRCz4IEkPi10qfvlaQOKxUyQs68yCyZB6AJBjHeCEgyLiw88Dri/HC2a1W5wJguUM9T2wB8RbwYONGW5Vii1v+IGTrB+5qrYlwj0zAitDMPioGIedjjpwYi1MDB2x2KizuhJ82HXF1IYU0cUuqihWCBabiBPPKR98+LBXUg5hHj9AN2MlNLDjpLykwnhjcfeBkYsTDfW5MwAbxNJKc49y+LnIdalH2qPBoYs5StY7GKp6X8jcKkpe/EHS1emIN+QVPch0dZLkD/6oBWOcmLgcszYGF5770invhEfpohx+gE3yMBLgAfjQHWcrqlTLTX4Xgs1ZcDhbO0E5EGHyNo9QJsTHQ1CdLAOOO3rOF6fbCigkSXoagJ5Dhk/SscbRK+mSppJV7OhMcVZZZGUO9C33GcIdo3rED+lmUMUk4tYXdwRqbzs4wpgFPzR8VfVZEBwhSiyHABlYZ9wROuzOxwgDzwUUJtYExcv1Z0lFLc2ywHeGQcAD8ucdibzCcoQCVWKRRDtGwagHErM6DdUlS5vJzQel8QPEJOhB82L3C/d6i40Iyk5iyMxhsaaQq156ZL6CnpxsBROAQaRk4Czvwo4XhSnRBtyzAUF6ZVbC0BrsaUXXxuwMx84CYeUAWKw70opGBWg0WeEX9AEmLwFFmCP1gCj62uxcXyxt6LADZyZwVU2jeknxvvkc2ChkPwmsUwztU9oy2L7PaN/tFUCNdNrAgDjeFRL9JBWuPDmApBHmnZaPevBrDfgNJM6JaFN8xM1jFOiTQLYXLF0nEfigESvhUJgZiGiloENuDWGPpHCgMKt3aVXGOxgF0hqgMHMGtwNSp4GEfZI5AewYLsQrqWIAKlezESnbeaw/ErUVWNKUJUQoGxyENBAa5+8h04wE8H1+XUtESMG0mLA0hI5F64juYrDN5KH1qylCyAlWhGjgQA5TcZPWudNz8Y1pS+p25ASfI/Yl8PEBF0TEefIYE0dsEkWDsMQt+VFj0HoGyUhJqWGU1uaXuYxSzNey2VOy4G1zoEXJE6YduTD1fRBDkmOPlhGFfkIk34c4SGdXkBR5ZkL7IOqCJEwyWJK8cjngaAonDGSPgF+cNwaVPfMoMhKT8ktic0ZjK7hyC9xoEbaGxUHS+7CjY+mRG6vcEIMezS0J7CHh1a4nKu0FBgdNJCofBe16bhQ3ax9w9H1HpA3Yo4LxYpzghi5PzQSQcYaoviVLEX/zyD/xJFJWNXBAtoMn2AMyfYLq0dDm5rN749X2fte82v7Dj2lnPOqfzlIC9ZuL9+l9IN+wMjSmkkqiBVcKbYsyj4gJsJFRKT8EG2DZjIhTTs6oGNVNpKghs7w4B6aLUs5km+tFARifYomSphOmAbvRYBZ/r0m46QXW7iXlkVtgWUDhE1Ivyanr+eDgZIkmQBkDzFryld6Ee48YQmE6XtmZlMQsDu4pN96NH9ZP/2mp8iDimJIRZC7hdHeOE3cvL3EQy+xxeR5unEuoaVGKLZTHRYJJ+STkc6f95KYiYTD5zmElKTmDdptXKBvrE51/WUa2k78KDtt8bbF+4HEo4DHz6xzCms+rOPdE66GaHmdENj/BU4tRiFAeQcJhJxjWs+gaaalqujoLqFYUFtPAZKjGPhWhzbFGJgoO2EUdMbGD24ns0/SGwRIRlDCdPAKRnCuT00/oOAtZFB8V8Wc4uC/MonWomI80lzpfWFEZHUxrQiXEHUkC/Ay7vnCZxkkZ4Dhi7zEU7NeVBTSMJ/gtAah3AlBB8qWvOgEoeRAuCdx0wlj3W5FbBOeiOhPWDcOr4gmqoBA3GSaJn7KQb3c6QzdxNlrp7vozlbGB+Gy0myU+KhJxGOVlu932CImhcqaBQe2TBtlTf1ls/HNYmwHPMw3Xx41GlyQccacslcL/4dWRt6TB2zZeXLmefMq2weOVoEiIvtj+4mUScAP2MHOhCxyJDF9weDbl/EnKXtCwTNwa8qJjMlByQ4IoWeTc/0ibYoME1iYHDimDM4fOB9l1V20+1WZIVwRnDYzsoCA+nccsK68OXr9ukaRIQplD0bJjo8mkOIIUBMAOFUTc7kqp1c6csFhU3bi3tYIitEH9c0V6lzcCuLUgM6dvcD4gRjP2TToqzcb0hCipKKKcovrxLFIE8THYiWSudhATU8mutL2yUUbxLVTEgoBl6nlE65CvKo8xV+gA01xAuaWASnQC06pRJIYenjqxSDdcF/6Xo4HGJG1DWOAJDug+Z1SPdTueKIS27p7ahU6VhaDvspvkjrxpyTEtPHjG+8q5YA/iR9cF1KEAGAH0vKkpMk9lDvePNCsgrMpkPdTUa6lrjGTr7s4AuhT3kAVVzMPeJaMcNH6SNfIUGwFkAURKEnJmWvioFzsSxlZJiodAMaautfYdDynXMy65RZpON0I8A9MRcXAVYRICGstgAATIM2kCKx5DLOBZ4+NPL0GJM8bONzPBUFKMUaC+OxGpNCiSCXs1LS/5HiDeNFHDoTyQwCc5kNzuEFWbhDl43ewToD8k+ErzVUHVmoG6BT/TIma3TsjZpbC8ponZGwbUD0ZLglWRWrgugtQniNv3WKIIInfLlG1OqGx3YsStWZ3WLTxI9sgBaz0ftD2QraE+ZoE4tLIALEMkLDcT1RygjKhHZOkWkDixA39FBxGAAQbOKbxKqEmPjCHHTZSSDdM0WbacaJAby5XaiZELg875fdkmfwjIZ1uKFbhZCnS0EiP3Zn/TDj0+dQQr79w/KuSGWNh8eO/6duAg3g7DsQAkLvoVxiCJN95kxEh4VIGcvmM5tmNXiEV3Bk2H2wRhl7JgiDUj1khYgsV9/iioxWKzxQVhYSvhgHEAd0EhaMoc49VOaoR5BkGQ/qGW2HkVxL0LU/N7H35T2TpltEOoStyQWnC8BI7XEQSqkzJqETYpQ2xLvLm7qz/Ys3na89lJONCsxeSGVXh0QV5Fb36nvCJeyyC0Tanug/JvV47YcZHUdyPD6WKxS1kcFwkK7EYKbIuau0cwRWCHqWPACRWym0APHW56EH5mXT2iquvuChEu3epjalvbTGoNMpoNBZhbP2ENtxa6N4wZ4Kah5JiHPJzA3WTvCiiCJnMqdAfkA6xlIUb6YRc2Fnbs2gR6iaV07o47lV5EG8UOz3ykCEzc1jtKyjjnjurANwgMQmAClMIJktlHrolc9elTEEceU8L42pYGhPgRnfyGLA4nit6c3PgQPO0BAEEQGUpakJJWrFKLETwa5JSI8n2zMugGCInbt5DlI7awv/zCBaOgRu6JgEYQZyPnnbL4AymApGG364RkJBSyJaxEIQYbsG3EB38JdSZPZSROHsYysQGY7dINMfBfQcwTM9E4YjjEotkBXhKmcnGjucSon33efXJA2d8G0iHZ43bcC09ASvPNxOkOMivdgH4MFBm1Chx/SykqQDuDD9v2VA5C4OcZwooxjDhfQFFhpQLiMVG2pLXB3LKBjCfH0vMrr2ArS8PqlcS3oJerJP7h3DvwfMs/ySajqpeAQwESLRLxVptCiJpc7TDBppA2xEHoYVeFSQLISGw+OA1VhlsnuH/l1AujUwJy66GCaQX7CD6wZTNKeVoc5Z784nAT5XXFyO4hqigKk4K4yFWFSfLMocW3uPUjgjIsnvvqE2Sni6grbnWK6gPhtBx87heGFWwC5Ra00PBCTo58aYtZ9ztDzgN7FXTmCo8nzyWpVJjOJBUtDeO9helQMAIKJ84vzFIScmhrsqi0Gr+wZQG395mVFakNghX9IUvLA18X+RDzfOBpw4kIQF7RjDByV5aGjIXE88o2eOBDgnoQyNmBUSI4S60QtGCycYCDA/bJIEoVwDSAf88ulsu7E6ucuO2XpHAhe9BKEgxgr2d0xqTdYUAql5V0DOQCQRgImSAKmY7lYFp6jESMTNhcXyQTtdcX+R0mQzFqWNPuixJkv/KruMtk85ulAEXkB24dQW5uJTrB9fYmo9QH9xOWTDpBA8seJY+BBYyYssjSJtOSojr19nD/xaBV7rG+anI76pbRjzG7LX1aZkrWbRaOQMWzNNrYyPFv64khmPKBML8tH4n6KFVSn8A5HkRmquo3Q0ET/SGWnXhMqO+LB4YgIXkuFF6jjZxd7C4z/gKWQf9304vh2tra9fhDcpnK7tHlj/ung8v914Ojs43Z+0U3uZpNP0w+7h3NpuPO87Uk/W+xHN8me8mnyXRjMl2qux5/HC831tX19W6y2Ulm8/T/d5Ob8XQjfeeL9H0vzg6Oj047yfdJr5O9ZD5e3s2n6dtvPz++Ho9v1R/mzRfqfcNO2ri8Yalf+NPB0f5hiZyfpm34V/au9Wwxpp9ef7F/cnKc/XV2/NP+2bH6q16S6l+ZDl/v5s8NXr/OrqVjof5XaTf1v7UIpzf+uxyd9+PF8nI5nqf9Ht1cTpbjT2n3fx1Pl7P5525yO59cjRfFCF39PFP/Sht4kf1b/ZeNVX5XNlzqBfVgJX9O1O//uBtNl5Pl52IEq4cT58Nd6tlE3dOp3vAhvZo/Vd03mSZ1D9RT//p357G6Z7FRPzf5kD2mbrZNQyrsRAOSvySb2VuG+nx/Gv22UYxON0mHdXR3s8xlyxDC6iNOQRxdLSezaTrKLsnt6DLrkte1orvZrT8kT3q7z6tRKFqff3Atu5zOxq+j5Th9afXC4lL6znQk8wHNX1oM8iQTieKu/BHtp/Sxi2H22EU+ZIufx9fN+9W11vtzwUhv3ag7l695/dZO1cir7GP1D/lLbkafx/PmGOVXm6OkdN5C73l2odUBdTW9K/vxIn/TUI3wZvLDXvm9HzItkd3SScY3i3HZKaVDL69md9Nl+orynvxrmXrVv9ZZK7twlQ1Z8YpcNV/m8qbEhFjCtaDXM3GxmTVTn7aiZeU6z57qaEJ4sZ5/bH2oFv260r3rw1KizHZMFsl0tkyUHNcCdtlV/1euoHzxFkNXPVndTH9SbSXr3cYiH1YP5YNzoX5WD+T/NJXJd9Vja/UU5B/LdFkxq0qTTK/HvymdMB9NP4436rnqPG8N6OdMpurBzZ/9LjUw8kEu/lVIgnanKQ+1XrXMZFsZ60qsHnxz4Jv9fDy6vR1PrzfKOewYd6YqZTmZ3o3RvDUmzPIBesY6nlOmS0a+QtRD+qfzWV2MbsaXy9lydLOodtBskNJ3Ps9Wv1Iy1B71Xb4NEc2oVYO+1zg2DWN/8f9ktcnkMjKbX4/nmQZYzObL8XW9urVO17vqL+PPezejT++vR0XnN4zJRhu29s58jrr8x81b0ZvUo9UV7dn5+NfxfDHeO5vfjbuETio2gUxBXKwr068St9Ynh8a0FSN58VyBIGvEFlgYZW8HRy8Pjl5fpovl7N3JIP2QWl7Fj6/2T34anFyevjl4dXaZ3lj+Wuz0y1R5XC4naTtGy0y7pwt3tpio9xdqJBUOpSoni8l0sRxNr8Yb5Q3dZONmslh2k+Xd7c24k42qUh/VC5I/7SX91ga+nrr/l8evLn88fnf08nQ9N9ZmN3ef0hfOZ/9UVu3oVllqWlOym1QzF4VCzjeh7EpryytavJFtceqNuVbLbu5kllL2S/5N/ceL9OZhp8NqcfFT/dxF/r7SkB99HOd7OGU0fbyZvR/dJOT0dRN64ta8bKi9vWRT2+Fc8mLeZJMbw+YzPQdkNnbMpv1lD9t3D8UeqrreXg35ormeXC07zr5k24EhvS0Titrw879sD2RP/EdykMpbcpMaqN9fjz4XBkvy8uB12qxFprcXyfLncXJ79/5mcpUsR7fj5H3qIkzHyc+zu3lXNXc0mRYvmywXavNOPzCf3aXG9Wx68zlJzYrJTfqS0TJRn0g3z8XjJDlLXzod/5Zfm48Xd5+KL6lPPC4HzCJQpA1W2f57lsdMq+E21b6T2Z0aKNNvVT7OZjd/l/J3h5qp7jDhTIEu317LU3pbZe6Wr1ECYrYKaF5jWJqLkRyVq3Tmp+ObrmrybF7rxW4y/u12fJXuspeFYiBf2hrdVCsYD5rNzyy17IPqxrLf5j1Zq+7m83HmGtQyXQ5S6+b0pWqz2CgfygdRfaDsTKaZqQ2peKRDG45Og/zHdweHL8thWB8aD45vymWfjWqx6rM1lu8Szcvlquw8t/Yue/oie5DZOfMJZw/1NhRPODvp1Pa6c1Y10jqTxY2q6UrIiH40XlWp+o2Gx9RYS3uuDmSghqlpi0YUCte4U/s5/8gvqV+Tf2L9fDB4me/b2uSREpNqTa0F5YL5z6S/rTas/iZzpW9kj/3Xf6nnUsdKmarpX6kmgrNj2r9l47rZrHVzKWuOdaHnvtPeXnjjypwsdMZ8/I+71GEbK5wo1d+pqp6nG4Yh2BfPtSUw7BhaWaCt1H/v5+PRL/rz9ef/REz5c7eH55A7Y/0YHk0tOJrEZJJiFxG7T6ovOFNG+JOZTWi5iRfT0lYbxGzWXkV7fIG7SboNp8cHhxpUp8wMZa6Wv2jg7kZ53FSC1ebpSnEc1ykefHN8cvD346P0uV5x5X/e7R+dHZz97fLVSb41q98ebxa/pq86OFQjdnx0+Lf0l1epJzQufnvxZvDir2+PD47ydvS3u8n2027ypN9Nnu2q469NFSiQXu3tPu1UDT9IW5B+7Xxw8PrNmQZOK1T5ebL1uF+gzhnc/DzpP94sLmgItX45h6rVkzsKhi6adnz06uDl4OiFmuKyLyeD/x68OBu8bHTjMJWFy9Ozwdv0+ve90u1azCY3l0oCZ+//r6t84AYQW7lemhRnt9J2ZvpTJs/6iwyIP/1xtFzO25/rGC1KPc9cgS6y1VV86ENmCv4zw6rrlufLr+2Caa1TD2ZLVz2p3tC8wb1U07t/Hi2ydueXi+Xa3h0/T8Y3+bI2upPbu5eLycfpKB2Gsd6pDCkzwZf1Uh09TzZr/379xfHx28alXFeYl7Ljj8aT+UmIebE4FDEvGtKn/1DKn9mg8+bjbwaDZhtfHx+fal35d70hlFPimvN8U518Sr3UPU1ci3nIfyk2pVoxXc1nt9Tt6nrrZjWX1M3ZHJc36wq9aM5kWsxdc89Q1y7ym4bJd3ua+swsvaxx7ofVLdSjWVPdj6pbikc1gyntX4aNtJdNta1r+LnuJeaP1Rtg9k/TuCqns7LeyPVJ4PT1bvKPu9F1uqUuiS+2XnQ3vZld/ZI6DNVDLde1mK9C59Tvqo2KWj6rt9SX8jHpGoit9nPewGK8p6NP4xwtU3+prhdoY/57p1NCcbo+uLu9Tk2ey6ubVLYur8fLsdpwc4RCbbkNcKah6ruJqedrP85U/2mTsv07gwmmSWszayrvNQO90HeGRSEn7YE2cJHmIzQ8Mru9TXudeWu91A7Nb6r6oOEdP+iQXWtrUygn2YHZP6eXFLZifNy8o7w6LDHxTwoNvZ3dzD5+NmwnizYvP1kvh7096716C/QloF7yKf1JffHDzWxkjGfxgXRQs1tsY9p4AbGGjM+3X6e1JxuF8n2j94uNuoHfNz7YUb6rMqYqWM0YQWXp1m9rzqlmw1QKz0Qy3HNPWkL6YstXmbI5fx1vFEurpRnUImksK92la31EuWDEwlYnF0r6i8/Ux8KttWE5HS4aVt9e3Jg+0jrtNSyM7GxB+3SO9hNfbp8F0x/O78ub2jgjNr6cTerl7Wgy177+QLVI0cnN1KpW/8+ySzRXn6FGiCXYBU9W6sX6rClE499SV2wy/XipjmXSOVWnL4tKcBepsaydaqdvyW6ozmkWhrfcOMTSjHh1b+71mQ6dEnc1ftkNHSXoW63fs9/UwbjyV7OTpNrff946rVJYePZAz/RV875UJ5ptFzW/wTgwa93zXVIgrBkyn32mP8zHtkO4rcVk529u7MlXs0/1qGv7cDrz06lOJRhnBqOiPBmWhcluqqWi8KJNzzT1qrtrDYDkw53aInIEtTpOL33wrvqsNrrXk3wmlbbXnvs++5xhl1XyYY6wiVNr7xg2SCPmYqUmuTjnaqORDHnjyBxD7tq3KUpBafa0TpqJXhiAD1uIK57THlcQSxJVYZcV0tVuS/FDtT4uyhmnCB00/t96TTZ2puS67u2Sd1XtaP3aAV/vZZ6JySspVmRxo7EkP8zT6bic3003csSomzRNZEPDaYd51bV0/M1198NeYeUUv2YbP2EnNDFa3cSsbG0DNqrvrSylbtsyIzdNXWPXZtZfGk8/b2hPzebNFL+iRikQT8e324u4o2+Sxd6iRq23SXe2IJ417BuTatayQXJA8GY+Hl1/vlyMb25SxVrdR+9vTa7ksNaLufLfqOW/EJbLUpI6pnazK3SD0agjDL+OJjej9xm4q5axMdKbbSqHhaFi3Pd9cwCMBzQKh4EtVA35wTgwp5WTrnzSNV09TIzQmkVhGV/otXv6abT8+fHVeHLT1hjVm/6cWDBWU92SfW50oSZPVZ3RwJdMwir3xsqn0e+fzfOXbzSOa3VGlvrvzyZ3qr5MArraR1O/x5z375KN1lafCkMpvZ3G+82mdBr0toohtlH2pUkR63TWdEJA8Ri9lst3KohiQ+cL5Z+9bL67Iq8pFqSmaWramkEjarDWCOJR/nhGJJLQUxreV/s3w2kzfqlgb4O2Yki+tr5bOyPlkTSp7cYDNM29RdiqJ61ix1SAjaIcmG1/7vZ2rY5yjv+v1b/phwCVncjDpNYaLOvG0U2bZY33cOJM6P8BtZzbv3ZHAgA='}


def unpack(payload):
    return gzip.decompress(base64.b64decode(payload.encode('ascii'))).decode('utf-8')

CONTROL_NAMES = tuple(CONTROL_BLOBS)
for idx, (name, payload) in enumerate(CONTROL_BLOBS.items()):
    source = unpack(payload)
    # Force the intended public agent to be the last callable selected by Kaggle.
    source = source.rstrip() + f"\n\nregression_entry_{idx} = agent\n"
    ast.parse(source)
    path = AGENT_DIR / f'control_{idx}.py'
    path.write_text(source, encoding='utf-8')
    PATHS[name] = path

print({'controls': CONTROL_NAMES})

{'controls': ('Kaito V21', 'Syouya V23', 'Zero-Waste V24', 'Soil V25')}


## 4. Loader contract and static tests

In [4]:
def last_callable(source):
    namespace = {}
    exec(compile(source, '<agent>', 'exec'), namespace)
    values = [(name, value) for name, value in namespace.items() if callable(value)]
    assert values
    return values[-1]

parent_name, parent_callable = last_callable(PARENT_SOURCE)
assert parent_callable.__code__.co_argcount in (1, 2)

for name, source in CANDIDATE_SOURCES.items():
    entry_name, entry = last_callable(source)
    assert entry_name == 'submission_agent'
    assert entry.__code__.co_argcount == 1

print({
    'parent_entry': parent_name,
    'candidate_entries': {name: last_callable(src)[0] for name,src in CANDIDATE_SOURCES.items()},
})

{'parent_entry': '_kaggle_submission_entrypoint', 'candidate_entries': {'Terminal only': 'submission_agent', 'Pressure low': 'submission_agent', 'Pressure medium': 'submission_agent'}}


## 5. Deterministic terminal test

In [5]:
namespace = {}
exec(compile(CANDIDATE_SOURCES['Terminal only'], '<terminal>', 'exec'), namespace)
policy = namespace['submission_agent']

farm = {
    'money': 100000,
    'tiles': [[None] * 10 for _ in range(10)],
    'farmer': [4, 4],
    'hands': [[4, 4]] * 6,
    'unlocked_quadrants': ['NW','NE','SW','SE'],
    'hires_today': 0,
}
obs = {
    'step': 718,
    'player': 0,
    'farms': [farm, dict(farm)],
    'private': {
        'shed': {'WHEAT': 8, 'WOOL': 3, 'MILK': 2},
        'seeds': {},
        'inventories': [{} for _ in range(7)],
    },
    'market': {
        'inventory': {item: 10000 for item in namespace['_MARKET_PARAMS']},
        'prices': {item: params[0] for item,params in namespace['_MARKET_PARAMS'].items()},
    },
    'town': {'unlocked_shops': []},
}
terminal = policy(obs)
assert ['SELL','WHEAT',8] in terminal['market']
assert ['SELL','WOOL',3] in terminal['market']
assert ['SELL','MILK',2] in terminal['market']
assert all(order[0] == 'SELL' for order in terminal['market'])
print(terminal['market'])

[['SELL', 'WHEAT', 8], ['SELL', 'MILK', 2], ['SELL', 'WOOL', 3]]


## 6. Official Kaggle environment

In [6]:
REQUIRED = '1.32.4'
try:
    installed = importlib.metadata.version('kaggle-environments')
except Exception:
    installed = None

if installed != REQUIRED:
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
        f'kaggle-environments=={REQUIRED}'
    ], check=True)
    importlib.invalidate_caches()

import kaggle_environments
from kaggle_environments import make
print({'engine': kaggle_environments.__version__})

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 29.9 MB/s eta 0:00:00


OpenSpiel exception: Unknown game 'go_fish'. Available games are:
2048
add_noise
amazons
ant_foraging_arena
antichess
backgammon
backgammon_proxy
banqi
bargaining
battleship
blackjack
blotto
breakthrough
bridge
bridge_arena
bridge_uncontested_bidding
cached_tree
catch
checkers
chess
chinese_checkers
cliff_walking
clobber
clobber_proxy
coin_game
coin_game_arena
colored_trails
connect_four
coop_box_pushing
coop_to_1p
coordinated_mp
crazy_eights
crazyhouse
cribbage
cursor_go
dark_chess
dark_hex
dark_hex_ir
deep_sea
dots_and_boxes
dots_and_boxes_proxy
dou_dizhu
efg_game
einstein_wurfelt_nicht
euchre
first_sealed_auction
gin_rummy
gin_rummy_proxy
go
gomoku
goofspiel
hanabi
havannah
havannah_proxy
hearts
hex
hive
kriegspiel
kuhn_poker
laser_tag
latent_ttt
leduc_poker
lewis_signaling
liars_dice
liars_dice_ir
lines_of_action
lines_of_action_proxy
maedn
mancala
mancala_proxy
markov_soccer
matching_pennies_3p
matrix_bos
matrix_brps
matrix_cd
matrix_coordination
matrix_mp
matrix_pd
matrix_rps
mat

{'engine': '1.32.4'}


OpenSpiel exception: Unknown game 'capture_the_flag'. Available games are:
2048
add_noise
amazons
amazons_proxy
ant_foraging_arena
antichess
backgammon
backgammon_proxy
banqi
bargaining
bargaining_proxy
battleship
blackjack
blotto
breakthrough
breakthrough_proxy
bridge
bridge_arena
bridge_uncontested_bidding
cached_tree
catch
chat_game
checkers
checkers_proxy
chess
chinese_checkers
cliff_walking
clobber
clobber_proxy
coin_game
coin_game_arena
coin_game_proxy
colored_trails
connect_four
connect_four_proxy
coop_box_pushing
coop_to_1p
coordinated_mp
crazy_eights
crazyhouse
cribbage
cursor_go
dark_chess
dark_hex
dark_hex_ir
dark_hex_proxy
deep_sea
dots_and_boxes
dots_and_boxes_proxy
dou_dizhu
efg_game
einstein_wurfelt_nicht
euchre
first_sealed_auction
gin_rummy
gin_rummy_proxy
go
go_proxy
gomoku
goofspiel
hanabi
havannah
havannah_proxy
hearts
hex
hive
hive_proxy
kriegspiel
kuhn_poker
laser_tag
latent_ttt
leduc_poker
lewis_signaling
liars_dice
liars_dice_ir
lines_of_action
lines_of_action_p

## 7. Symmetric evaluator

In [7]:
def money(env, seat):
    final = env.steps[-1][seat]
    try:
        return float(final.reward)
    except Exception:
        pass
    for states in reversed(env.steps):
        try:
            return float(states[0].observation.farms[seat].money)
        except Exception:
            continue
    return float('-inf')


def play(left, right, seed):
    env = make('kaggriculture', configuration={'episodeSteps':720, 'seed':int(seed)}, debug=False)
    env.run([str(PATHS[left]), str(PATHS[right])])
    final = env.steps[-1]
    assert str(final[0].status) == 'DONE'
    assert str(final[1].status) == 'DONE'
    return money(env,0), money(env,1)


def symmetric(candidate, opponent, seed):
    a0,a1 = play(candidate, opponent, seed)
    b0,b1 = play(opponent, candidate, seed)
    return [
        {'candidate':candidate,'opponent':opponent,'seed':seed,'seat':0,'money':a0,'opp_money':a1,'margin':a0-a1},
        {'candidate':candidate,'opponent':opponent,'seed':seed,'seat':1,'money':b1,'opp_money':b0,'margin':b1-b0},
    ]

smoke = play('Parent', CONTROL_NAMES[0], 26080701)
print({'smoke': smoke})

{'smoke': (147324.0, 83140.0)}


## 8. Stage A — direct challenge against exact v22

In [8]:
DIRECT_SEEDS = (26080711, 26080723, 26080737, 26080751)
direct_rows = []
for candidate in CANDIDATE_SOURCES:
    for seed in DIRECT_SEEDS:
        direct_rows.extend(symmetric(candidate, 'Parent', seed))


def summarize(rows, name):
    group = [row for row in rows if row['candidate'] == name]
    margins = [row['margin'] for row in group]
    return {
        'candidate': name,
        'games': len(group),
        'wins': sum(x > 0 for x in margins),
        'ties': sum(x == 0 for x in margins),
        'losses': sum(x < 0 for x in margins),
        'mean_margin': round(statistics.fmean(margins), 2),
        'minimum_margin': min(margins),
        'mean_money': round(statistics.fmean(row['money'] for row in group), 2),
    }

direct_summary = [summarize(direct_rows, name) for name in CANDIDATE_SOURCES]
print(json.dumps(direct_summary, indent=2))
FINALISTS = [r['candidate'] for r in direct_summary if r['losses']==0 and r['wins']>0 and r['mean_margin']>0][:2]
print({'finalists': FINALISTS})

[
  {
    "candidate": "Terminal only",
    "games": 8,
    "wins": 2,
    "ties": 6,
    "losses": 0,
    "mean_margin": 75.0,
    "minimum_margin": 0.0,
    "mean_money": 117705.75
  },
  {
    "candidate": "Pressure low",
    "games": 8,
    "wins": 0,
    "ties": 0,
    "losses": 8,
    "mean_margin": -2137.25,
    "minimum_margin": -2717.0,
    "mean_money": 116493.25
  },
  {
    "candidate": "Pressure medium",
    "games": 8,
    "wins": 0,
    "ties": 0,
    "losses": 8,
    "mean_margin": -2485.0,
    "minimum_margin": -2786.0,
    "mean_money": 116292.75
  }
]
{'finalists': ['Terminal only']}


## 9. Stage B — regression gate

In [9]:
REGRESSION_SEEDS = (26080801, 26080819)
regression_rows = []
for candidate in ['Parent', *FINALISTS]:
    for opponent in CONTROL_NAMES:
        for seed in REGRESSION_SEEDS:
            regression_rows.extend(symmetric(candidate, opponent, seed))

regression_summary = []
for candidate in ['Parent', *FINALISTS]:
    group = [row for row in regression_rows if row['candidate'] == candidate]
    margins = [row['margin'] for row in group]
    regression_summary.append({
        'candidate': candidate,
        'games': len(group),
        'wins': sum(x > 0 for x in margins),
        'ties': sum(x == 0 for x in margins),
        'losses': sum(x < 0 for x in margins),
        'mean_margin': round(statistics.fmean(margins), 2),
        'minimum_margin': min(margins),
        'minimum_money': min(row['money'] for row in group),
    })
print(json.dumps(regression_summary, indent=2))

[
  {
    "candidate": "Parent",
    "games": 16,
    "wins": 16,
    "ties": 0,
    "losses": 0,
    "mean_margin": 32170.44,
    "minimum_margin": 11269.0,
    "minimum_money": 98160.0
  },
  {
    "candidate": "Terminal only",
    "games": 16,
    "wins": 16,
    "ties": 0,
    "losses": 0,
    "mean_margin": 32268.56,
    "minimum_margin": 11524.0,
    "minimum_money": 98425.0
  }
]


## 10. Promotion and submission

In [10]:
def by_name(rows, name):
    return next(row for row in rows if row['candidate'] == name)

SELECTED_NAME = 'Parent'
SELECTED_SOURCE = PARENT_SOURCE

if FINALISTS:
    parent_reg = by_name(regression_summary, 'Parent')
    eligible = []
    for name in FINALISTS:
        direct = by_name(direct_summary, name)
        broad = by_name(regression_summary, name)
        if (
            direct['losses'] == 0
            and direct['wins'] > 0
            and direct['mean_margin'] > 0
            and broad['wins'] >= parent_reg['wins']
            and broad['losses'] <= parent_reg['losses']
            and broad['mean_margin'] >= parent_reg['mean_margin']
            and broad['minimum_margin'] >= parent_reg['minimum_margin']
            and broad['minimum_money'] >= parent_reg['minimum_money']
        ):
            eligible.append((broad['wins'], -broad['losses'], direct['mean_margin'], broad['mean_margin'], name))
    if eligible:
        eligible.sort(reverse=True)
        SELECTED_NAME = eligible[0][-1]
        SELECTED_SOURCE = CANDIDATE_SOURCES[SELECTED_NAME]

MAIN_PATH = WORK / 'main.py'
ARCHIVE_PATH = WORK / 'submission.tar.gz'
MAIN_PATH.write_text(SELECTED_SOURCE, encoding='utf-8')

entry_name, entry = last_callable(SELECTED_SOURCE)
assert entry.__code__.co_argcount in (1,2)

raw = io.BytesIO()
with gzip.GzipFile(fileobj=raw, mode='wb', filename='', mtime=0) as gz:
    with tarfile.open(fileobj=gz, mode='w') as tar:
        payload = SELECTED_SOURCE.encode('utf-8')
        info = tarfile.TarInfo('main.py')
        info.size = len(payload)
        info.mode = 0o644
        info.mtime = 0
        info.uid = info.gid = 0
        info.uname = info.gname = ''
        tar.addfile(info, io.BytesIO(payload))
ARCHIVE_PATH.write_bytes(raw.getvalue())

with tarfile.open(ARCHIVE_PATH, 'r:gz') as tar:
    assert tar.getnames() == ['main.py']
    packed = tar.extractfile('main.py').read().decode('utf-8')
assert packed == SELECTED_SOURCE

print({
    'selected': SELECTED_NAME,
    'fallback_to_exact_v22': SELECTED_NAME == 'Parent',
    'resolved_callable': entry_name,
    'main_sha256': hashlib.sha256(SELECTED_SOURCE.encode()).hexdigest(),
    'submission': str(ARCHIVE_PATH),
    'ready': True,
})

{'selected': 'Terminal only', 'fallback_to_exact_v22': False, 'resolved_callable': 'submission_agent', 'main_sha256': 'fd75c6710c655f761fb9cd3eaa4e9a192583e08719812711539090bd109ac71b', 'submission': '/kaggle/working/submission.tar.gz', 'ready': True}


## Output

Run **Restart Session → Run All** and submit:

```text
/kaggle/working/submission.tar.gz
```

If neither extension clears both gates, the archive is the exact attached v22 agent
(SHA-256 `62fb5a5f66f0011092a2b51e3192879ba583d5d815d761f176091c615657147a`).